# ETF Sector Rotation Strategy

RSI + LogBias momentum rotation across A-share sector / commodity / dividend ETFs.

Train: 2009-01-01 ~ 2019-12-31 (search window split into train 2009-2016 / validation 2017-2019).  
Out-of-sample: 2020-01-01 ~ present.


## 1. Imports, Constants, And Config


In [ ]:
from __future__ import annotations
import joblib

import os

for k in [
    "HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy",
    "ALL_PROXY", "all_proxy"
]:
    os.environ.pop(k, None)

os.environ["NO_PROXY"] = "*"
os.environ["no_proxy"] = "*"


from dataclasses import dataclass
from itertools import product
from pathlib import Path
from typing import Dict, Optional
import os
import shutil
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import akshare as ak
import tushare as ts

warnings.filterwarnings("ignore")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

TUSHARE_TOKEN = os.environ.get("TUSHARE_TOKEN", "")
START_DATE = "2009-01-01"
TRAIN_END_DATE = "2019-12-31"
TRAIN_SEARCH_START_DATE = "2009-01-01"
TRAIN_SEARCH_END_DATE = "2016-12-31"
VALIDATION_START_DATE = "2017-01-01"
VALIDATION_END_DATE = "2019-12-31"
OOS_START_DATE = "2020-01-01"
END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")
BENCHMARK_CODE = "510300.SH"
BENCHMARK_NAME = "沪深300ETF华泰柏瑞"
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "ETF_Result"
N_WORKERS = os.cpu_count() or 1

EMA_WINDOW_RANGE = [25, 30, 35]
RSI_ENTRY_RANGE = [46, 48]
RSI_EXIT_RANGE = [48, 50]
CATEGORY_ENTRY_SHIFT_RANGE = [-1.2, -0.6]
CATEGORY_SOFT_SHIFT_RANGE = [-0.6, -0.3, 0.0]
CATEGORY_STOP_SHIFT_RANGE = [-0.5, 0.0]
STRONG_DAYS_SHIFT_RANGE = [-1, 0]
USE_RELATIVE_STRENGTH_FILTER_RANGE = [False]
CANDIDATE_EXPOSURE_MAP_OPTIONS = {
    "base": ((5, 1.00), (4, 0.96), (3, 0.90), (2, 0.78), (1, 0.60), (0, 0.00)),
    "balanced": ((5, 0.88), (4, 0.82), (3, 0.74), (2, 0.60), (1, 0.42), (0, 0.00)),
    "aggressive": ((5, 1.00), (4, 1.00), (3, 0.98), (2, 0.92), (1, 0.80), (0, 0.00)),
}
EXPOSURE_MAP_VERSION_RANGE = ["base", "balanced", "aggressive"]
MAX_DRAWDOWN_LIMIT = -0.15
TRAIN_DRAWDOWN_BUDGET = -0.20
TEST_DRAWDOWN_TARGET = -0.12
VALIDATION_DRAWDOWN_CAP = -0.12
TARGET_OOS_DRAWDOWN = -0.15
TARGET_OOS_ANNUAL_RETURN = 0.2503

BEST_OOS_BASELINE_PARAMS = {
    "ema_window": 25,
    "rsi_entry": 44.0,
    "rsi_exit": 50.0,
    "stock_entry_shift": -1.2,
    "commodity_entry_shift": -1.2,
    "dividend_entry_shift": -1.2,
    "stock_soft_shift": -0.6,
    "commodity_soft_shift": -0.3,
    "dividend_soft_shift": -0.6,
    "stock_stop_shift": -0.5,
    "commodity_stop_shift": -0.5,
    "dividend_stop_shift": -0.5,
    "strong_days_shift": -1,
    "use_relative_strength_filter": False,
    "exposure_map_version": "base",
    "portfolio_exposure_cap": 1.0,
    "dd_limit_1": -0.24,
    "dd_limit_2": -0.28,
    "dd_limit_3": -0.36,
    "dd_cap_1": 0.98,
    "dd_cap_2": 0.92,
    "dd_cap_3": 0.85,
    "defensive_mode": "cash",
    "defensive_allocation_cap": 0.0,
    "defensive_trigger_dd": -0.10,
}

LIGHT_RISK_EXPOSURE_MAP_RANGE = ["base", "balanced", "aggressive"]
LIGHT_RISK_PORTFOLIO_CAP_RANGE = [0.80, 0.85, 0.90, 0.95, 1.00]
LIGHT_RISK_DD_LIMIT_SETS = [
    (-0.18, -0.22, -0.28),
    (-0.20, -0.24, -0.30),
    (-0.08, -0.12, -0.16),
    (-0.10, -0.14, -0.18),
    (-0.12, -0.16, -0.20),
]
LIGHT_RISK_DD_CAP_SETS = [
    (0.92, 0.82, 0.70),
    (0.90, 0.78, 0.64),
    (0.85, 0.70, 0.55),
    (0.88, 0.74, 0.60),
    (0.95, 0.86, 0.74),
]
DEFENSIVE_MODES = ["cash", "bond"]
DEFENSIVE_ALLOCATION_CAP_RANGE = [0.20, 0.30, 0.40, 0.50]
DEFENSIVE_TRIGGER_DD_RANGE = [-0.08, -0.10, -0.12]

TREND_ETF_POOL: Dict[str, str] = {
    "电池ETF广发": "159755.SZ", "新能源车ETF华夏": "515030.SH", "半导体ETF国联安": "512480.SH", "航空航天ETF华夏": "159227.SZ",
    "电网设备ETF华夏": "159326.SZ", "游戏ETF华夏": "159869.SZ", "房地产ETF": "512200.SZ", "银行ETF华宝": "512800.SH",
    "光伏ETF": "159857.SZ", "机器人ETF华夏": "562500.SH", "家电ETF富国": "561120.SH", "中证红利ETF招商": "515080.SH",
    "建材ETF富国": "516750.SH", "金融科技ETF华宝": "159851.SZ", "创新药ETF": "159992.SZ", "人工智能ETF": "515980.SH",
    "软件ETF国泰": "515230.SH", "通信ETF国泰": "515880.SH", "消费电子ETF": "159779.SZ", "卫星ETF富国": "563230.SH",
    "稀土ETF富国": "159713.SZ", "化工ETF": "159870.SZ", "油气ETF博时": "561760.SH", "有色ETF大成": "159980.SZ",
    "云计算ETF招商": "159890.SZ", "证券ETF": "159841.SZ", "酒ETF鹏华": "512690.SH", "畜牧ETF": "159867.SZ",
    "钢铁ETF国泰": "515210.SH", "煤炭ETF国泰": "515220.SH", "证券ETF国泰": "512880.SH", "影视ETF": "159855.SZ",
    "石油ETF国泰": "561360.SH", "豆粕ETF华夏": "159985.SZ", "红利低波ETF富国": "159525.SZ", "黄金ETF易方达": "159934.SZ",
    "创业板ETF广发": "159952.SZ",
    "可转债ETF博时": "511380.SH", "信用债ETF博时": "159396.SZ", "公司债ETF平安": "511030.SH", "短融ETF海富通": "511360.SH",
    "十年国债ETF国泰": "511260.SH", "30年国债ETF鹏扬": "511090.SH", "国债ETF国泰": "511010.SH",
}

SYMBOL_NAME_MAP: Dict[str, str] = {code: name for name, code in TREND_ETF_POOL.items()}
SYMBOL_NAME_MAP[BENCHMARK_CODE] = BENCHMARK_NAME

ASSET_CATEGORY_MAP: Dict[str, str] = {
    "159755.SZ": "stock", "515030.SH": "stock", "512480.SH": "stock", "159227.SZ": "stock", "159326.SZ": "stock",
    "159869.SZ": "stock", "512200.SZ": "stock", "512800.SH": "dividend", "159857.SZ": "stock", "562500.SH": "stock",
    "561120.SH": "stock", "515080.SH": "dividend", "516750.SH": "stock", "159851.SZ": "stock", "159992.SZ": "stock",
    "515980.SH": "stock", "515230.SH": "stock", "515880.SH": "stock", "159779.SZ": "stock", "563230.SH": "stock",
    "159713.SZ": "stock", "159870.SZ": "stock", "561760.SH": "stock", "159980.SZ": "stock", "159890.SZ": "stock",
    "159841.SZ": "stock", "512690.SH": "stock", "159867.SZ": "stock", "515210.SH": "commodity", "515220.SH": "commodity",
    "512880.SH": "stock", "159855.SZ": "stock", "561360.SH": "commodity", "159985.SZ": "commodity", "159934.SZ": "commodity",
    "511380.SH": "bond", "159396.SZ": "bond", "511030.SH": "bond", "511360.SH": "bond", "511260.SH": "bond",
    "511090.SH": "bond", "511010.SH": "bond", "159525.SZ": "dividend", "159952.SZ": "stock", 
}

OPTIMIZED_CATEGORIES = ("stock", "commodity", "dividend")
THREE_CLASS_MAP = {k: v for k, v in ASSET_CATEGORY_MAP.items() if v in OPTIMIZED_CATEGORIES}
BOND_CLASS_MAP = {k: v for k, v in ASSET_CATEGORY_MAP.items() if v == "bond"}
ENTRY = {"stock": 4.0, "commodity": 2.8, "dividend": 1.0}
STOP = {"stock": -5.5, "commodity": -4.8, "dividend": -3.0}
OVERHEAT = {"stock": 16.5, "commodity": 11.0, "dividend": 6.0}
SOFT = {"stock": 0.8, "commodity": 0.3, "dividend": 0.0}
STRONG = {"stock": 3, "commodity": 2, "dividend": 2}


@dataclass
class StrategyConfig:
    ema_window: int = 20
    top_n: int = 5
    fee_rate: float = 0.0003
    slippage_rate: float = 0.0005
    stamp_duty_rate: float = 0.0
    initial_capital: float = 1_000_000.0
    dynamic_holdings: bool = True
    min_holdings: int = 3
    max_holdings: int = 5
    dd_limit_1: float = -0.24
    dd_limit_2: float = -0.28
    dd_limit_3: float = -0.36
    dd_cap_1: float = 0.98
    dd_cap_2: float = 0.92
    dd_cap_3: float = 0.85
    portfolio_exposure_cap: float = 1.0
    enable_daily_stop: bool = True
    enable_overheat_trim: bool = True
    trim_ratio: float = 0.15
    candidate_exposure_map: tuple = CANDIDATE_EXPOSURE_MAP_OPTIONS["base"]
    use_relative_strength_filter: bool = True
    strong_days_required_shift: int = 0
    exposure_map_version: str = "base"
    score_weight_logbias: float = 0.4
    score_weight_slope: float = 0.2
    score_weight_ret20: float = 0.2
    score_weight_relative_strength: float = 0.2
    defensive_mode: str = "cash"
    defensive_allocation_cap: float = 0.0
    defensive_trigger_dd: float = -0.10


## 2. Data Loading


In [ ]:
def normalize_symbol(symbol: str) -> str:
    symbol = str(symbol).strip().upper()
    if "." in symbol:
        return symbol
    if symbol.startswith(("159", "399", "000")):
        return f"{symbol}.SZ"
    return f"{symbol}.SH"
def get_symbol_label(symbol: str) -> str:
    code = normalize_symbol(symbol)
    return f"{SYMBOL_NAME_MAP.get(code, code)} ({code})"
def repair_price_series_with_pct_chg(df: pd.DataFrame, ts_code: str) -> pd.DataFrame:
    out = df.copy().sort_values("date").reset_index(drop=True)
    if "pct_chg" not in out.columns:
        return out

    out["pct_chg"] = pd.to_numeric(out["pct_chg"], errors="coerce")
    raw_close_return = out["close"].pct_change()
    pct_chg_return = out["pct_chg"] / 100.0
    mismatch = (raw_close_return - pct_chg_return).abs()
    anomaly_mask = (mismatch > 0.20) & pct_chg_return.notna() & raw_close_return.notna()
    if not anomaly_mask.any():
        return out.drop(columns=["pct_chg"])

    adjusted_close = []
    for idx, row in out.iterrows():
        if idx == 0:
            adjusted_close.append(float(row["close"]))
            continue
        pct_ret = row["pct_chg"] / 100.0 if pd.notna(row["pct_chg"]) else raw_close_return.iloc[idx]
        if pd.isna(pct_ret):
            pct_ret = raw_close_return.iloc[idx]
        if pd.isna(pct_ret):
            pct_ret = 0.0
        adjusted_close.append(adjusted_close[-1] * (1.0 + float(pct_ret)))

    out["adjusted_close"] = adjusted_close
    scale = out["adjusted_close"] / out["close"].replace(0, np.nan)
    for col in ["open", "high", "low", "close"]:
        out[col] = out[col] * scale
    anomaly_dates = out.loc[anomaly_mask, "date"].dt.strftime("%Y-%m-%d").tolist()
    print(f"  [DATA FIX] {get_symbol_label(ts_code)} repaired split-like price jumps on {', '.join(anomaly_dates[:5])}")
    if len(anomaly_dates) > 5:
        print(f"  [DATA FIX] {get_symbol_label(ts_code)} additional repaired dates: {len(anomaly_dates) - 5}")
    return out.drop(columns=["pct_chg", "adjusted_close"])
def load_akshare_daily(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    ts_code = normalize_symbol(symbol)
    raw_symbol = ts_code.split(".")[0]
    df = ak.fund_etf_hist_em(
        symbol=raw_symbol,
        period="daily",
        start_date=start_date.replace("-", ""),
        end_date=end_date.replace("-", ""),
        adjust="qfq",
    )
    rename_map = {
        "日期": "date",
        "开盘": "open",
        "最高": "high",
        "最低": "low",
        "收盘": "close",
        "成交量": "volume",
        "涨跌幅": "pct_chg",
    }
    out = df.rename(columns=rename_map)
    keep_cols = ["date", "open", "high", "low", "close", "volume", "pct_chg"]
    out = out[[col for col in keep_cols if col in out.columns]].copy()
    out["date"] = pd.to_datetime(out["date"])
    for col in [c for c in ["open", "high", "low", "close", "volume", "pct_chg"] if c in out.columns]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.sort_values("date").dropna(subset=["open", "high", "low", "close"])
    out = out[(out["date"] >= pd.Timestamp(start_date)) & (out["date"] <= pd.Timestamp(end_date))]
    out = repair_price_series_with_pct_chg(out, ts_code)
    out["symbol"] = ts_code
    out["is_trading"] = True
    return out.reset_index(drop=True)
def load_tushare_daily(symbol: str, start_date: str, end_date: str, source_type: str = "fund") -> pd.DataFrame:
    pro = ts.pro_api(TUSHARE_TOKEN)
    ts_code = normalize_symbol(symbol)
    start = start_date.replace("-", "")
    end = end_date.replace("-", "")
    try:
        if source_type == "map":
            df = pro.index_daily(ts_code=ts_code, start_date=start, end_date=end)
        else:
            df = pro.fund_daily(ts_code=ts_code, start_date=start, end_date=end)
        df = df.rename(columns={"trade_date": "date", "vol": "volume"})
        keep_cols = ["date", "open", "high", "low", "close", "volume"]
        if "pct_chg" in df.columns:
            keep_cols.append("pct_chg")
        out = df[keep_cols].copy()
        out["date"] = pd.to_datetime(out["date"])
        for col in [c for c in ["open", "high", "low", "close", "volume", "pct_chg"] if c in out.columns]:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        out = out.sort_values("date").dropna(subset=["open", "high", "low", "close"])
        out = out[(out["date"] >= pd.Timestamp(start_date)) & (out["date"] <= pd.Timestamp(end_date))]
        out = repair_price_series_with_pct_chg(out, ts_code)
        out["symbol"] = ts_code
        out["is_trading"] = True
        return out.reset_index(drop=True)
    except Exception as exc:
        if source_type == "fund":
            print(f"  [DATA FALLBACK] {get_symbol_label(ts_code)} tushare unavailable, fallback to akshare: {exc}")
            return load_akshare_daily(ts_code, start_date, end_date)
        raise
def align_market_data(data_dict: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    all_dates = sorted(set().union(*[set(df["date"]) for df in data_dict.values()]))
    master_index = pd.DatetimeIndex(all_dates, name="date")
    aligned = {}
    for symbol, df in data_dict.items():
        tmp = df.set_index("date").reindex(master_index)
        tmp["symbol"] = symbol
        tmp["is_trading"] = tmp["is_trading"].fillna(False).infer_objects(copy=False)
        tmp["close"] = tmp["close"].ffill()
        tmp["volume"] = tmp["volume"].fillna(0.0)
        aligned[symbol] = tmp.reset_index()
    return aligned


## 3. Indicators And Metrics


In [ ]:
def calculate_indicators(df: pd.DataFrame, ema_window: int, slope_lookback: int = 5) -> pd.DataFrame:
    out = df.copy()
    out["log_close"] = np.log(out["close"])
    out["log_ema"] = out["log_close"].ewm(span=ema_window, adjust=False, min_periods=ema_window).mean()
    out["logbias"] = (out["log_close"] - out["log_ema"]) * 100
    out["price_ema"] = out["close"].ewm(span=ema_window, adjust=False, min_periods=ema_window).mean()
    out["ret_20"] = out["close"].pct_change(20)
    out["logbias_slope"] = out["logbias"] - out["logbias"].shift(slope_lookback)
    return out
def calc_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - 100 / (1 + rs)
    rsi = rsi.where(avg_loss.ne(0), 100)
    rsi = rsi.where(avg_gain.ne(0), 0)
    return rsi
def build_benchmark_curve(benchmark_df: pd.DataFrame) -> pd.DataFrame:
    out = benchmark_df[["date", "close"]].dropna().sort_values("date").copy()
    out["benchmark_nav"] = out["close"] / out["close"].iloc[0]
    return out[["date", "benchmark_nav"]]
def build_drawdown_series(nav: pd.Series) -> pd.Series:
    return nav / nav.cummax() - 1.0
def calculate_equity_diagnostics(equity_curve: pd.DataFrame) -> Dict[str, float]:
    if equity_curve is None or equity_curve.empty:
        return {
            "avg_exposure": 0.0,
            "peak_exposure": 0.0,
            "days_exposure_above_0_9": 0,
            "worst_month_return": 0.0,
            "worst_window_avg_exposure": 0.0,
        }

    out = equity_curve.copy()
    out["drawdown"] = build_drawdown_series(out["nav"])
    out["month"] = out["date"].dt.to_period("M")
    monthly = out.groupby("month")["nav"].agg(["first", "last"])
    monthly_return = monthly["last"] / monthly["first"] - 1.0

    worst_idx = out["drawdown"].idxmin()
    peak_idx = out.loc[:worst_idx, "nav"].idxmax()
    worst_window = out.loc[peak_idx:worst_idx]

    return {
        "avg_exposure": float(out["exposure"].mean()),
        "peak_exposure": float(out["exposure"].max()),
        "days_exposure_above_0_9": int((out["exposure"] > 0.9).sum()),
        "worst_month_return": float(monthly_return.min()) if not monthly_return.empty else 0.0,
        "worst_window_avg_exposure": float(worst_window["exposure"].mean()) if not worst_window.empty else 0.0,
    }
def calculate_performance_metrics(
    equity_curve: pd.DataFrame,
    trades: pd.DataFrame,
    initial_capital: float,
    benchmark_curve: Optional[pd.DataFrame],
) -> Dict[str, float]:
    if equity_curve.empty:
        return {
            "total_return": 0.0,
            "annual_return": 0.0,
            "max_drawdown": 0.0,
            "sharpe_ratio": -np.inf,
            "calmar_ratio": 0.0,
            "win_rate": np.nan,
            "avg_hold_days": np.nan,
            "turnover": 0.0,
            "avg_exposure": 0.0,
            "min_exposure": 0.0,
            "excess_return": np.nan,
            "annual_excess_return": np.nan,
        }

    nav = equity_curve["nav"]
    daily_ret = nav.pct_change().fillna(0.0)
    total_days = max(len(nav), 1)
    total_return = float(nav.iloc[-1] - 1.0)
    annual_return = float(nav.iloc[-1] ** (252 / total_days) - 1.0) if total_days > 1 else 0.0
    max_drawdown = float(build_drawdown_series(nav).min()) if len(nav) > 0 else 0.0
    sharpe = float(daily_ret.mean() / daily_ret.std(ddof=0) * np.sqrt(252)) if daily_ret.std(ddof=0) > 0 else -np.inf
    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else 0.0
    closed_trades = trades[trades["exit_date"].notna()].copy() if not trades.empty else pd.DataFrame()
    win_rate = float((closed_trades["trade_return"] > 0).mean()) if not closed_trades.empty else np.nan
    avg_hold_days = float(closed_trades["holding_days"].mean()) if not closed_trades.empty else np.nan
    turnover = float(equity_curve["turnover"].sum() / max(equity_curve["equity"].mean(), initial_capital))

    metrics = {
        "total_return": total_return,
        "annual_return": annual_return,
        "max_drawdown": max_drawdown,
        "sharpe_ratio": sharpe if not np.isnan(sharpe) else -np.inf,
        "calmar_ratio": calmar,
        "win_rate": win_rate,
        "avg_hold_days": avg_hold_days,
        "turnover": turnover,
        "avg_exposure": float(equity_curve["exposure"].mean()),
        "min_exposure": float(equity_curve["exposure"].min()),
        "excess_return": np.nan,
        "annual_excess_return": np.nan,
    }

    if benchmark_curve is not None and not benchmark_curve.empty:
        merged = equity_curve[["date", "nav"]].merge(benchmark_curve, on="date", how="left")
        merged["benchmark_nav"] = merged["benchmark_nav"].ffill().bfill()
        excess_curve = merged["nav"] / merged["benchmark_nav"]
        metrics["excess_return"] = float(excess_curve.iloc[-1] - 1.0)
        metrics["annual_excess_return"] = float(excess_curve.iloc[-1] ** (252 / len(merged)) - 1.0) if len(merged) > 1 else 0.0

    return metrics
def calculate_panel_diagnostics(panel: Dict[str, pd.DataFrame], equity_curve: pd.DataFrame) -> Dict[str, float]:
    if not panel:
        return {
            "hard_signal_days_ratio": 0.0,
            "soft_signal_days_ratio": 0.0,
            "invested_days_ratio": 0.0,
            "avg_candidate_count": 0.0,
            "avg_soft_candidate_count": 0.0,
        }

    hard_counts = []
    soft_counts = []
    all_dates = sorted(set().union(*[set(df["date"]) for df in panel.values()]))
    for date in all_dates:
        hard_count = 0
        soft_count = 0
        for df in panel.values():
            row = df[df["date"] == date]
            if row.empty:
                continue
            row = row.iloc[0]
            if bool(row["rotation_candidate"]):
                hard_count += 1
            if bool(row["soft_candidate"]):
                soft_count += 1
        hard_counts.append(hard_count)
        soft_counts.append(soft_count)

    invested_days_ratio = float((equity_curve["exposure"] > 0.01).mean()) if not equity_curve.empty else 0.0
    return {
        "hard_signal_days_ratio": float(np.mean(np.array(hard_counts) > 0)) if hard_counts else 0.0,
        "soft_signal_days_ratio": float(np.mean(np.array(soft_counts) > 0)) if soft_counts else 0.0,
        "invested_days_ratio": invested_days_ratio,
        "avg_candidate_count": float(np.mean(hard_counts)) if hard_counts else 0.0,
        "avg_soft_candidate_count": float(np.mean(soft_counts)) if soft_counts else 0.0,
    }
def calculate_param_distance(params: Dict[str, object]) -> float:
    baseline = BEST_OOS_BASELINE_PARAMS
    distance = 0.0
    numeric_keys = [
        "ema_window",
        "rsi_entry",
        "rsi_exit",
        "stock_entry_shift",
        "commodity_entry_shift",
        "dividend_entry_shift",
        "stock_soft_shift",
        "commodity_soft_shift",
        "dividend_soft_shift",
        "stock_stop_shift",
        "commodity_stop_shift",
        "dividend_stop_shift",
        "strong_days_shift",
    ]
    for key in numeric_keys:
        distance += abs(float(params[key]) - float(baseline[key]))
    categorical_keys = ["use_relative_strength_filter", "exposure_map_version"]
    for key in categorical_keys:
        distance += 1.0 if params[key] != baseline[key] else 0.0
    return distance
def build_category_shift_map(stock_value: float, commodity_value: float, dividend_value: float) -> Dict[str, float]:
    return {
        "stock": float(stock_value),
        "commodity": float(commodity_value),
        "dividend": float(dividend_value),
    }
def calc_trade_price(open_price: float, slippage_rate: float, side: str) -> float:
    return open_price * (1 + slippage_rate) if side == "buy" else open_price * (1 - slippage_rate)
def calc_transaction_cost(amount: float, fee_rate: float, stamp_duty_rate: float, side: str) -> float:
    cost = amount * fee_rate
    if side == "sell":
        cost += amount * stamp_duty_rate
    return cost


## 4. Backtester


In [ ]:
class RSIRotationBacktester:
    def __init__(self, config: StrategyConfig):
        self.config = config

    @staticmethod
    def _build_market_view(panel: Dict[str, pd.DataFrame]) -> Dict[pd.Timestamp, Dict[str, Dict]]:
        market_view = {}
        for symbol, df in panel.items():
            for _, row in df.iterrows():
                market_view.setdefault(row["date"], {})
                market_view[row["date"]][symbol] = row.to_dict()
        return market_view

    @staticmethod
    def _should_rebalance(date: pd.Timestamp, next_date: pd.Timestamp) -> bool:
        current_iso, next_iso = date.isocalendar(), next_date.isocalendar()
        return (current_iso.year, current_iso.week) != (next_iso.year, next_iso.week)

    def _candidate_based_exposure(self, hard_count: int) -> float:
        for min_count, exposure in self.config.candidate_exposure_map:
            if hard_count >= min_count:
                return exposure
        return 0.0

    def _get_drawdown_exposure_cap(self, equity: float, equity_peak: float) -> float:
        if equity_peak <= 0:
            return self.config.portfolio_exposure_cap
        dd = equity / equity_peak - 1.0
        if dd <= self.config.dd_limit_3:
            return min(self.config.dd_cap_3, self.config.portfolio_exposure_cap)
        if dd <= self.config.dd_limit_2:
            return min(self.config.dd_cap_2, self.config.portfolio_exposure_cap)
        if dd <= self.config.dd_limit_1:
            return min(self.config.dd_cap_1, self.config.portfolio_exposure_cap)
        return self.config.portfolio_exposure_cap

    def _get_defensive_allocation_cap(self, equity: float, equity_peak: float) -> float:
        if self.config.defensive_mode not in {"cash", "bond"}:
            return 0.0
        if equity_peak <= 0:
            return 0.0
        dd = equity / equity_peak - 1.0
        if dd <= self.config.defensive_trigger_dd:
            return self.config.defensive_allocation_cap
        return 0.0

    def _generate_target_weights(
        self,
        daily_map: Dict[str, Dict],
        open_trades: Dict[str, Dict],
        exposure_cap: float,
        defensive_cap: float,
    ) -> Dict[str, float]:
        ranked, soft_ranked = [], []
        for symbol, row in daily_map.items():
            if not row["is_trading"]:
                continue
            if bool(row.get("rotation_candidate", False)):
                ranked.append((symbol, row["rotation_score"]))
            elif bool(row.get("soft_candidate", False)):
                soft_ranked.append((symbol, row["rotation_score"]))

        ranked.sort(key=lambda x: x[1], reverse=True)
        soft_ranked.sort(key=lambda x: x[1], reverse=True)

        desired_count = self.config.top_n
        hard_count = len(ranked)
        if self.config.dynamic_holdings:
            if hard_count > 0:
                desired_count = min(max(hard_count, self.config.min_holdings), self.config.max_holdings)
            elif open_trades:
                desired_count = min(max(len(open_trades), self.config.min_holdings), self.config.max_holdings)
            else:
                desired_count = 0

        selected = []
        for symbol, _ in ranked:
            if len(selected) >= desired_count:
                break
            selected.append(symbol)

        for symbol in list(open_trades.keys()):
            if len(selected) >= desired_count:
                break
            if symbol not in selected and symbol in daily_map:
                row = daily_map[symbol]
                keep_cond = row.get("logbias", -99.0) > row.get("category_stop_threshold", -5.0) and row.get("rsi14", 0.0) > 47
                if keep_cond:
                    selected.append(symbol)

        for symbol, _ in soft_ranked:
            if len(selected) >= desired_count:
                break
            if symbol not in selected:
                selected.append(symbol)

        if not selected:
            target_weights = {}
        else:
            target_exposure = min(self._candidate_based_exposure(hard_count), exposure_cap)
            target_weights = {symbol: target_exposure / len(selected) for symbol in selected}

        if self.config.defensive_mode != "bond" or defensive_cap <= 0:
            return target_weights

        allocated_weight = float(sum(target_weights.values()))
        available_defensive_weight = min(max(exposure_cap - allocated_weight, 0.0), defensive_cap)
        if available_defensive_weight <= 0:
            return target_weights

        defensive_symbols = [
            symbol for symbol, row in daily_map.items()
            if symbol in BOND_CLASS_MAP and row.get("is_trading", False) and pd.notna(row.get("open"))
        ]
        if not defensive_symbols:
            return target_weights

        defensive_weight = available_defensive_weight / len(defensive_symbols)
        for symbol in defensive_symbols:
            target_weights[symbol] = defensive_weight
        return target_weights

    def _close_trade(self, open_trade: Dict, exit_date: pd.Timestamp, exit_price: float, exit_reason: str) -> Dict:
        trade = dict(open_trade)
        trade["exit_date"] = exit_date
        trade["exit_price"] = exit_price
        trade["exit_reason"] = exit_reason
        trade["holding_days"] = int((exit_date - trade["entry_date"]).days)
        trade["trade_return"] = exit_price / trade["entry_price"] - 1.0
        return trade

    @staticmethod
    def _build_buy_signal_snapshot(
        signal_date: pd.Timestamp,
        execution_date: pd.Timestamp,
        daily_map: Dict[str, Dict],
        target_weights: Dict[str, float],
    ) -> Optional[Dict[str, object]]:
        allocations = []
        for symbol, weight in sorted(target_weights.items(), key=lambda x: x[1], reverse=True):
            if weight <= 0:
                continue
            row = daily_map.get(symbol, {})
            allocations.append(
                {
                    "symbol": symbol,
                    "name": SYMBOL_NAME_MAP.get(symbol, symbol),
                    "category": ASSET_CATEGORY_MAP.get(symbol, ""),
                    "weight": float(weight),
                    "rotation_candidate": bool(row.get("rotation_candidate", False)),
                    "soft_candidate": bool(row.get("soft_candidate", False)),
                }
            )
        if not allocations:
            return None
        return {
            "signal_date": signal_date,
            "execution_date": execution_date,
            "allocations": allocations,
        }

    @staticmethod
    def _compute_weight_map(daily_map: Dict[str, Dict], positions: Dict[str, float], cash: float) -> Dict[str, float]:
        position_values = {}
        total_equity = float(cash)
        for symbol, qty in positions.items():
            row = daily_map.get(symbol)
            if row is None or pd.isna(row.get("close")):
                continue
            value = float(qty * row["close"])
            if value <= 1e-12:
                continue
            position_values[symbol] = value
            total_equity += value
        if total_equity <= 0:
            return {}
        return {symbol: value / total_equity for symbol, value in position_values.items()}

    def _build_trade_plan_snapshot(
        self,
        signal_date: pd.Timestamp,
        execution_date: pd.Timestamp,
        daily_map: Dict[str, Dict],
        positions: Dict[str, float],
        open_trades: Dict[str, Dict],
        cash: float,
        should_weekly_rotate: bool,
        weekly_target_weights: Dict[str, float],
        exposure_cap: Optional[float],
        defensive_cap: Optional[float],
    ) -> Dict[str, object]:
        current_weights = self._compute_weight_map(daily_map, positions, cash)
        risk_target_weights = dict(current_weights)
        trigger_types = {}
        summary_triggers = []
        adjusted_open_trades = dict(open_trades)

        for symbol in list(current_weights.keys()):
            row = daily_map.get(symbol)
            if row is None or not row.get("is_trading", False) or pd.isna(row.get("close")):
                continue

            exit_rsi = row.get("exit_rsi_threshold", 45)
            should_clear = (
                row.get("logbias", np.nan) < row.get("category_stop_threshold", -5.0)
                or (row.get("close", np.nan) < row.get("price_ema", np.nan) and row.get("rsi14", 100.0) < exit_rsi)
            )
            soft_trim = bool(
                self.config.enable_overheat_trim
                and (
                    (row.get("rsi14", 0.0) > 78 and not bool(row.get("rsi_up", True)))
                    or (
                        row.get("logbias", 0.0) > row.get("category_overheat_threshold", 15.0)
                        and row.get("logbias_slope", 0.0) < 0
                    )
                )
            )

            if should_clear:
                risk_target_weights[symbol] = 0.0
                trigger_types.setdefault(symbol, set()).add("hard_exit")
                adjusted_open_trades.pop(symbol, None)
                if "hard_exit" not in summary_triggers:
                    summary_triggers.append("hard_exit")
            elif soft_trim:
                risk_target_weights[symbol] = current_weights.get(symbol, 0.0) * (1.0 - self.config.trim_ratio)
                trigger_types.setdefault(symbol, set()).add("soft_trim")
                if "soft_trim" not in summary_triggers:
                    summary_triggers.append("soft_trim")

        final_target_weights = dict(risk_target_weights)
        if should_weekly_rotate:
            final_target_weights = dict(weekly_target_weights)
            if "weekly_rebalance" not in summary_triggers:
                summary_triggers.append("weekly_rebalance")
            for symbol, symbol_triggers in trigger_types.items():
                if "hard_exit" in symbol_triggers:
                    final_target_weights[symbol] = 0.0
                elif "soft_trim" in symbol_triggers:
                    final_target_weights[symbol] = min(
                        final_target_weights.get(symbol, 0.0),
                        risk_target_weights.get(symbol, 0.0),
                    )

        all_symbols = set(current_weights) | set(final_target_weights)
        rows = []
        for symbol in all_symbols:
            current_weight = float(current_weights.get(symbol, 0.0))
            target_weight = float(final_target_weights.get(symbol, 0.0))
            if current_weight <= 1e-10 and target_weight <= 1e-10:
                continue

            delta_weight = target_weight - current_weight
            symbol_trigger_types = set(trigger_types.get(symbol, set()))
            if should_weekly_rotate:
                symbol_trigger_types.add("weekly_rebalance")

            if current_weight <= 1e-10 and target_weight > 1e-10:
                action = f"\u8c03\u5165\uff0c\u589e\u52a0 {target_weight:.2%}"
            elif current_weight > 1e-10 and target_weight <= 1e-10:
                action = f"\u9000\u51fa\uff0c\u51cf\u5c11 {abs(delta_weight):.2%}"
            elif delta_weight > 1e-10:
                action = f"\u52a0\u4ed3\uff0c\u589e\u52a0 {delta_weight:.2%}"
            elif delta_weight < -1e-10:
                action = f"\u51cf\u4ed3\uff0c\u51cf\u5c11 {abs(delta_weight):.2%}"
            else:
                action = "\u6301\u6709\u4e0d\u53d8"

            rows.append(
                {
                    "symbol": symbol,
                    "name": SYMBOL_NAME_MAP.get(symbol, symbol),
                    "current_weight": current_weight,
                    "target_weight": target_weight,
                    "delta_weight": delta_weight,
                    "action": action,
                    "trigger_type": ",".join(sorted(symbol_trigger_types)) if symbol_trigger_types else "none",
                }
            )

        rows.sort(key=lambda item: (-item["target_weight"], -item["current_weight"], item["symbol"]))
        delta_weights = {row["symbol"]: row["delta_weight"] for row in rows}
        actions = {row["symbol"]: row["action"] for row in rows}
        trigger_type_map = {row["symbol"]: row["trigger_type"] for row in rows}

        return {
            "signal_date": signal_date,
            "execution_date": execution_date,
            "current_weights": current_weights,
            "target_weights": {row["symbol"]: row["target_weight"] for row in rows},
            "delta_weights": delta_weights,
            "trigger_types": trigger_type_map,
            "actions": actions,
            "summary_triggers": summary_triggers,
            "rows": rows,
            "exposure_cap": exposure_cap,
            "defensive_cap": defensive_cap,
        }

    def _execute_target_weights(self, date, daily_map, cash, positions, target_weights, open_trades):
        closed_trades = []
        turnover_today = 0.0
        total_value = cash + sum(
            qty * (daily_map[s]["open"] if pd.notna(daily_map[s]["open"]) else daily_map[s]["close"])
            for s, qty in positions.items()
        )

        for symbol in list(positions.keys()):
            row = daily_map.get(symbol)
            if row is None or not row["is_trading"] or pd.isna(row["open"]):
                continue
            current_value = positions[symbol] * row["open"]
            target_value = total_value * target_weights.get(symbol, 0.0)
            diff = target_value - current_value
            if diff >= 0:
                continue
            sell_price = calc_trade_price(row["open"], self.config.slippage_rate, "sell")
            sell_qty = min(-diff / sell_price, positions[symbol])
            amount = sell_qty * sell_price
            cost = calc_transaction_cost(amount, self.config.fee_rate, self.config.stamp_duty_rate, "sell")
            cash += amount - cost
            turnover_today += amount
            positions[symbol] -= sell_qty
            if positions[symbol] <= 1e-8:
                closed_trades.append(self._close_trade(open_trades.pop(symbol), date, sell_price, "rotation_rebalance"))
                del positions[symbol]

        for symbol, weight in target_weights.items():
            row = daily_map.get(symbol)
            if row is None or not row["is_trading"] or pd.isna(row["open"]):
                continue
            target_value = total_value * weight
            current_value = positions.get(symbol, 0.0) * row["open"]
            diff = target_value - current_value
            if diff <= 0:
                continue
            buy_price = calc_trade_price(row["open"], self.config.slippage_rate, "buy")
            available_amount = cash / (1 + self.config.fee_rate)
            buy_amount = min(diff, available_amount)
            if buy_amount <= 0:
                continue
            buy_qty = buy_amount / buy_price
            amount = buy_qty * buy_price
            cost = calc_transaction_cost(amount, self.config.fee_rate, self.config.stamp_duty_rate, "buy")
            if amount + cost > cash:
                continue
            cash -= amount + cost
            turnover_today += amount
            positions[symbol] = positions.get(symbol, 0.0) + buy_qty
            if symbol not in open_trades:
                open_trades[symbol] = {
                    "symbol": symbol,
                    "entry_date": date,
                    "entry_price": buy_price,
                    "entry_reason": "defensive_entry" if symbol in BOND_CLASS_MAP else "rotation_entry",
                }

        return cash, positions, turnover_today, closed_trades

    def _apply_daily_risk_controls(self, date, daily_map, cash, positions, open_trades):
        closed = []
        for symbol in list(positions.keys()):
            row = daily_map.get(symbol)
            if row is None or not row["is_trading"] or pd.isna(row.get("open")):
                continue
            qty = positions[symbol]
            if qty <= 0:
                continue

            exit_rsi = row.get("exit_rsi_threshold", 45)
            should_clear = (
                row.get("logbias", np.nan) < row.get("category_stop_threshold", -5.0)
                or (row.get("close", np.nan) < row.get("price_ema", np.nan) and row.get("rsi14", 100.0) < exit_rsi)
            )
            soft_trim = bool(
                self.config.enable_overheat_trim
                and (
                    (row.get("rsi14", 0.0) > 78 and not bool(row.get("rsi_up", True)))
                    or (
                        row.get("logbias", 0.0) > row.get("category_overheat_threshold", 15.0)
                        and row.get("logbias_slope", 0.0) < 0
                    )
                )
            )
            if not should_clear and not soft_trim:
                continue

            ratio = 1.0 if should_clear else self.config.trim_ratio
            sell_qty = qty * ratio
            sell_price = calc_trade_price(row["open"], self.config.slippage_rate, "sell")
            amount = sell_qty * sell_price
            cost = calc_transaction_cost(amount, self.config.fee_rate, self.config.stamp_duty_rate, "sell")
            cash += amount - cost
            positions[symbol] -= sell_qty
            if positions[symbol] <= 1e-8:
                closed.append(self._close_trade(open_trades.pop(symbol), date, sell_price, "rsi_exit" if should_clear else "rsi_trim"))
                del positions[symbol]

        return cash, positions, closed

    def run(
        self,
        panel: Dict[str, pd.DataFrame],
        benchmark_df: Optional[pd.DataFrame] = None,
        defensive_panel: Optional[Dict[str, pd.DataFrame]] = None,
    ):
        merged_panel = dict(panel)
        if defensive_panel:
            merged_panel.update(defensive_panel)
        market_view = self._build_market_view(merged_panel)
        dates = sorted(market_view.keys())
        cash = self.config.initial_capital
        equity_peak = self.config.initial_capital
        positions = {}
        open_trades = {}
        pending_weights = None
        latest_buy_signal = None
        latest_trade_plan = None
        trade_records = []
        equity_records = []

        for idx, date in enumerate(dates):
            daily_map = market_view[date]
            turnover_today = 0.0

            if self.config.enable_daily_stop and positions:
                cash, positions, daily_closed = self._apply_daily_risk_controls(date, daily_map, cash, positions, open_trades)
                trade_records.extend(daily_closed)

            if pending_weights is not None:
                cash, positions, rebalance_turnover, closed = self._execute_target_weights(
                    date, daily_map, cash, positions, pending_weights, open_trades
                )
                turnover_today += rebalance_turnover
                trade_records.extend(closed)
                pending_weights = None

            equity = cash + sum(qty * daily_map[s]["close"] for s, qty in positions.items())
            equity_peak = max(equity_peak, equity)
            exposure = 0.0 if equity <= 0 else max(equity - cash, 0.0) / equity
            equity_records.append(
                {
                    "date": date,
                    "equity": equity,
                    "nav": equity / self.config.initial_capital,
                    "cash": cash,
                    "position_value": equity - cash,
                    "exposure": exposure,
                    "turnover": turnover_today,
                }
            )

            next_trade_date = dates[idx + 1] if idx < len(dates) - 1 else pd.Timestamp(date) + pd.offsets.BDay(1)
            should_weekly_rotate = False
            if idx < len(dates) - 1 and self._should_rebalance(date, dates[idx + 1]):
                should_weekly_rotate = True
            elif pd.Timestamp(date).weekday() == 4:
                should_weekly_rotate = True

            exposure_cap = None
            defensive_cap = None
            weekly_target_weights = {}
            if should_weekly_rotate:
                exposure_cap = self._get_drawdown_exposure_cap(equity, equity_peak)
                defensive_cap = self._get_defensive_allocation_cap(equity, equity_peak)
                weekly_target_weights = self._generate_target_weights(daily_map, open_trades, exposure_cap, defensive_cap)
                pending_weights = weekly_target_weights
                latest_buy_signal = self._build_buy_signal_snapshot(
                    signal_date=date,
                    execution_date=next_trade_date,
                    daily_map=daily_map,
                    target_weights=weekly_target_weights,
                )

            latest_trade_plan = self._build_trade_plan_snapshot(
                signal_date=date,
                execution_date=next_trade_date,
                daily_map=daily_map,
                positions=positions,
                open_trades=open_trades,
                cash=cash,
                should_weekly_rotate=should_weekly_rotate,
                weekly_target_weights=weekly_target_weights,
                exposure_cap=exposure_cap,
                defensive_cap=defensive_cap,
            )

        if open_trades and dates:
            last_map = market_view[dates[-1]]
            last_date = dates[-1]
            for symbol, trade in list(open_trades.items()):
                trade_records.append(self._close_trade(trade, last_date, last_map[symbol]["close"], "forced_close"))

        equity_curve = pd.DataFrame(equity_records)
        trades_df = pd.DataFrame(trade_records)
        benchmark_curve = build_benchmark_curve(benchmark_df) if benchmark_df is not None else None
        metrics = calculate_performance_metrics(equity_curve, trades_df, self.config.initial_capital, benchmark_curve)
        return equity_curve, trades_df, metrics, latest_buy_signal, latest_trade_plan


## 5. Signal Panel And Operation Guide


In [ ]:
def prepare_panel_rsi_only(
    aligned: Dict[str, pd.DataFrame],
    benchmark_df: pd.DataFrame,
    config: StrategyConfig,
    rsi_entry: float,
    rsi_exit: float,
    entry_shifts: Dict[str, float],
    soft_shifts: Dict[str, float],
    stop_shifts: Dict[str, float],
) -> Dict[str, pd.DataFrame]:
    bench = benchmark_df[["date", "close"]].copy().sort_values("date")
    bench["benchmark_ret20"] = bench["close"].pct_change(20)

    panel = {}
    for symbol, df in aligned.items():
        category = THREE_CLASS_MAP.get(symbol)
        if category is None:
            continue

        tmp = calculate_indicators(df, ema_window=config.ema_window)
        tmp["rsi14"] = calc_rsi(tmp["close"], 14)
        tmp["rsi_up"] = tmp["rsi14"] > tmp["rsi14"].shift(1)
        tmp = tmp.merge(bench[["date", "benchmark_ret20"]], on="date", how="left")
        tmp["benchmark_ret20"] = tmp["benchmark_ret20"].ffill().fillna(0.0)
        tmp["relative_strength"] = tmp["ret_20"] - tmp["benchmark_ret20"]
        tmp["category_entry_threshold"] = ENTRY[category] + entry_shifts[category]
        tmp["category_stop_threshold"] = STOP[category] + stop_shifts[category]
        tmp["category_overheat_threshold"] = OVERHEAT[category]
        tmp["category_soft_entry_threshold"] = SOFT[category] + soft_shifts[category]
        min_strong_days = max(1, STRONG[category] + config.strong_days_required_shift)
        tmp["strong_days_10"] = (tmp["logbias"] > tmp["category_entry_threshold"]).rolling(10, min_periods=1).sum()
        relative_strength_filter = tmp["relative_strength"].gt(0) if config.use_relative_strength_filter else True
        tmp["rotation_candidate"] = (
            tmp["logbias"].gt(tmp["category_entry_threshold"])
            & tmp["strong_days_10"].ge(min_strong_days)
            & tmp["close"].gt(tmp["price_ema"])
            & relative_strength_filter
            & tmp["logbias"].le(tmp["category_overheat_threshold"])
            & tmp["rsi14"].gt(rsi_entry)
            & tmp["is_trading"]
        )
        tmp["soft_candidate"] = (
            tmp["close"].gt(tmp["price_ema"])
            & tmp["logbias"].gt(tmp["category_soft_entry_threshold"])
            & tmp["rsi14"].gt(max(50, rsi_entry - 2))
            & tmp["is_trading"]
        )
        tmp["exit_rsi_threshold"] = rsi_exit
        tmp["rotation_score"] = (
            config.score_weight_logbias * tmp["logbias"]
            + config.score_weight_slope * tmp["logbias_slope"]
            + config.score_weight_ret20 * (tmp["ret_20"] * 100)
            + config.score_weight_relative_strength * (tmp["relative_strength"] * 100)
            + 0.10 * (tmp["rsi14"] - 50)
        )
        panel[symbol] = tmp[
            [
                "date",
                "open",
                "close",
                "is_trading",
                "price_ema",
                "logbias",
                "ret_20",
                "logbias_slope",
                "rsi14",
                "rsi_up",
                "relative_strength",
                "strong_days_10",
                "category_entry_threshold",
                "category_stop_threshold",
                "category_overheat_threshold",
                "category_soft_entry_threshold",
                "exit_rsi_threshold",
                "rotation_candidate",
                "soft_candidate",
                "rotation_score",
            ]
        ].copy()
    return panel
def prepare_defensive_bond_panel(aligned: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    panel = {}
    for symbol, df in aligned.items():
        if symbol not in BOND_CLASS_MAP:
            continue
        panel[symbol] = df[["date", "open", "close", "is_trading"]].copy()
    return panel
def build_top_scored_targets(panel: Dict[str, pd.DataFrame], top_k: int = 10) -> pd.DataFrame:
    rows = []
    for symbol, df in panel.items():
        if df is None or df.empty:
            continue
        latest_row = df.dropna(subset=["rotation_score"]).tail(1)
        if latest_row.empty:
            continue
        row = latest_row.iloc[0]
        rows.append(
            {
                "date": row["date"],
                "symbol": symbol,
                "name": SYMBOL_NAME_MAP.get(symbol, symbol),
                "category": THREE_CLASS_MAP.get(symbol, ""),
                "rotation_score": float(row["rotation_score"]),
                "rotation_candidate": bool(row["rotation_candidate"]),
                "soft_candidate": bool(row["soft_candidate"]),
                "logbias": float(row["logbias"]) if pd.notna(row["logbias"]) else np.nan,
                "rsi14": float(row["rsi14"]) if pd.notna(row["rsi14"]) else np.nan,
                "ret_20": float(row["ret_20"]) if pd.notna(row["ret_20"]) else np.nan,
                "relative_strength": float(row["relative_strength"]) if pd.notna(row["relative_strength"]) else np.nan,
            }
        )

    if not rows:
        return pd.DataFrame()

    summary = pd.DataFrame(rows).sort_values(
        by=["date", "rotation_score"],
        ascending=[False, False],
    )
    latest_date = summary["date"].max()
    summary = summary[summary["date"] == latest_date].copy()
    summary = summary.sort_values("rotation_score", ascending=False).head(top_k).reset_index(drop=True)
    summary.insert(0, "rank", np.arange(1, len(summary) + 1))
    return summary
def print_latest_buy_signal(latest_buy_signal: Optional[Dict[str, object]]) -> None:
    if not latest_buy_signal:
        print("\nLatest Buy Signal: none")
        return

    signal_date = pd.Timestamp(latest_buy_signal["signal_date"]).strftime("%Y-%m-%d")
    execution_date = pd.Timestamp(latest_buy_signal["execution_date"]).strftime("%Y-%m-%d")
    print(f"\nLatest Buy Signal: signal_date={signal_date}, execution_date={execution_date}")
    for idx, item in enumerate(latest_buy_signal["allocations"], start=1):
        candidate_flag = "hard" if item["rotation_candidate"] else ("soft" if item["soft_candidate"] else "hold")
        print(
            f"  {idx:>2}. {item['name']} ({item['symbol']}) | "
            f"weight={item['weight']:.2%} | category={item['category']} | type={candidate_flag}"
        )

def build_next_trade_holdings_table(trade_plan: Optional[Dict[str, object]]) -> pd.DataFrame:
    columns = ["symbol", "name", "current_weight", "target_weight", "delta_weight", "action", "trigger_type"]
    if not trade_plan or not trade_plan.get("rows"):
        return pd.DataFrame(columns=columns)
    df = pd.DataFrame(trade_plan["rows"])
    return df[columns].copy()

def print_signal_summary(trade_plan: Optional[Dict[str, object]]) -> None:
    print("\n今日触发信号摘要：")
    if not trade_plan:
        print("  无最新信号，维持现有仓位。")
        return

    signal_date = pd.Timestamp(trade_plan["signal_date"]).strftime("%Y-%m-%d")
    execution_date = pd.Timestamp(trade_plan["execution_date"]).strftime("%Y-%m-%d")
    summary_map = {
        "weekly_rebalance": "周度调仓",
        "hard_exit": "日度hard-exit",
        "soft_trim": "日度soft trim",
    }
    summary_triggers = trade_plan.get("summary_triggers", [])
    if summary_triggers:
        summary_text = "、".join(summary_map.get(item, item) for item in summary_triggers)
    else:
        summary_text = "无新信号，下一交易日维持现有仓位"

    print(f"  信号日：{signal_date}")
    print(f"  执行日：{execution_date}")
    print(f"  触发类型：{summary_text}")

def print_weight_change_details(trade_plan: Optional[Dict[str, object]]) -> None:
    print("\n仓位变动说明：")
    if not trade_plan or not trade_plan.get("rows"):
        print("  无持仓变化。")
        return

    rows = trade_plan["rows"]
    meaningful_rows = [row for row in rows if abs(float(row["delta_weight"])) > 1e-10]
    if not meaningful_rows:
        print("  今日无新增调仓，下一交易日维持当前持仓。")
        return

    for row in meaningful_rows:
        print(
            f"  {row['name']} ({row['symbol']})："
            f"当前 {row['current_weight']:.2%} -> 目标 {row['target_weight']:.2%}，"
            f"{row['action']}，触发类型={row['trigger_type']}"
        )

def print_next_trade_holdings_table(trade_plan: Optional[Dict[str, object]]) -> pd.DataFrame:
    print("\n下一交易日目标持仓表：")
    table = build_next_trade_holdings_table(trade_plan)
    if table.empty:
        print("  无持仓数据。")
        return table

    display_table = table.copy()
    for col in ["current_weight", "target_weight", "delta_weight"]:
        display_table[col] = display_table[col].map(lambda x: f"{x:.2%}")
    print(display_table.to_string(index=False))
    return table

def export_next_trade_holdings_csv(trade_plan: Optional[Dict[str, object]], output_dir: Path) -> Optional[Path]:
    table = build_next_trade_holdings_table(trade_plan)
    if trade_plan is None or table.empty:
        return None

    output_dir.mkdir(exist_ok=True, parents=True)
    signal_date = pd.Timestamp(trade_plan["signal_date"]).strftime("%Y%m%d")
    execution_date = pd.Timestamp(trade_plan["execution_date"]).strftime("%Y%m%d")
    csv_path = output_dir / f"next_trade_holdings_{signal_date}_to_{execution_date}.csv"
    export_table = table.copy()
    for col in ["current_weight", "target_weight", "delta_weight"]:
        export_table[col] = export_table[col].astype(float).round(6)
    export_table.to_csv(csv_path, index=False, encoding="utf-8-sig")
    return csv_path

def print_today_tomorrow_plan(
    latest_buy_signal: Optional[Dict[str, object]],
    as_of_date: pd.Timestamp,
) -> None:
    today = pd.Timestamp(as_of_date).normalize()

    print("\n今日/下一交易日持仓与操作：")
    print(f"  今日日期：{today.strftime('%Y-%m-%d')}")

    if not latest_buy_signal:
        next_trade_date = today + pd.offsets.BDay(1)
        print(f"  下一交易日：{next_trade_date.strftime('%Y-%m-%d')}")
        print("  今日持仓：无最新调仓信号，维持现有仓位。")
        print("  今日操作：无。")
        print("  下一交易日操作：若无新信号，继续维持现有仓位。")
        return

    signal_date = pd.Timestamp(latest_buy_signal["signal_date"]).normalize()
    execution_date = pd.Timestamp(latest_buy_signal["execution_date"]).normalize()
    allocations = latest_buy_signal["allocations"]
    holding_text = "；".join(
        f"{item['name']} ({item['symbol']}) {item['weight']:.2%}"
        for item in allocations
    )

    print(f"  下一交易日：{execution_date.strftime('%Y-%m-%d')}")

    if today >= execution_date:
        print(f"  今日持仓：{holding_text}")
        print(f"  今日操作：今日已处于 {execution_date.strftime('%Y-%m-%d')} 开盘调仓后的持仓，继续持有。")
        print("  下一交易日操作：若无新调仓信号，继续按当前持仓持有。")
    elif today == signal_date:
        print("  今日持仓：当前仍按上一期持仓收盘。")
        print(f"  今日操作：今日收盘后生成新调仓信号，计划于 {execution_date.strftime('%Y-%m-%d')} 开盘执行。")
        print(f"  下一交易日预期持仓：{holding_text}")
        print("  下一交易日操作：按上述目标仓位买入/调仓。")
    else:
        print(f"  最新信号日：{signal_date.strftime('%Y-%m-%d')}")
        print(f"  计划持仓：{holding_text}")
        print("  今日操作：等待执行。")
        print("  下一交易日操作：如到执行日开盘，则按目标仓位调仓；否则继续等待。")


## 6. Parameter Search


In [ ]:
def evaluate_single_params(args, train_aligned_data_list, train_benchmark_dict, validation_aligned_data_list, validation_benchmark_dict):
    (
        ema_window,
        rsi_entry,
        rsi_exit,
        stock_entry_shift,
        commodity_entry_shift,
        dividend_entry_shift,
        stock_soft_shift,
        commodity_soft_shift,
        dividend_soft_shift,
        stock_stop_shift,
        commodity_stop_shift,
        dividend_stop_shift,
        strong_days_shift,
        use_relative_strength_filter,
        exposure_map_version,
    ) = args

    try:
        config = StrategyConfig(
            ema_window=int(ema_window),
            candidate_exposure_map=CANDIDATE_EXPOSURE_MAP_OPTIONS[exposure_map_version],
            use_relative_strength_filter=bool(use_relative_strength_filter),
            strong_days_required_shift=int(strong_days_shift),
            exposure_map_version=exposure_map_version,
        )
        entry_shifts = build_category_shift_map(stock_entry_shift, commodity_entry_shift, dividend_entry_shift)
        soft_shifts = build_category_shift_map(stock_soft_shift, commodity_soft_shift, dividend_soft_shift)
        stop_shifts = build_category_shift_map(stock_stop_shift, commodity_stop_shift, dividend_stop_shift)

        def run_period_eval(aligned_data_list, benchmark_dict):
            aligned_data = {item["symbol"]: pd.DataFrame(item["data"]) for item in aligned_data_list}
            benchmark_df = pd.DataFrame(benchmark_dict)
            panel = prepare_panel_rsi_only(
                aligned_data,
                benchmark_df,
                config,
                rsi_entry=rsi_entry,
                rsi_exit=rsi_exit,
                entry_shifts=entry_shifts,
                soft_shifts=soft_shifts,
                stop_shifts=stop_shifts,
            )
            backtester = RSIRotationBacktester(config)
            equity_curve, trades, metrics, _, _ = backtester.run(panel, benchmark_df=benchmark_df)
            diagnostics = calculate_panel_diagnostics(panel, equity_curve)
            return equity_curve, trades, metrics, diagnostics

        _, train_trades, train_metrics, train_diagnostics = run_period_eval(train_aligned_data_list, train_benchmark_dict)
        _, validation_trades, validation_metrics, validation_diagnostics = run_period_eval(
            validation_aligned_data_list,
            validation_benchmark_dict,
        )
        candidate_params = {
            "ema_window": int(ema_window),
            "rsi_entry": rsi_entry,
            "rsi_exit": rsi_exit,
            "stock_entry_shift": stock_entry_shift,
            "commodity_entry_shift": commodity_entry_shift,
            "dividend_entry_shift": dividend_entry_shift,
            "stock_soft_shift": stock_soft_shift,
            "commodity_soft_shift": commodity_soft_shift,
            "dividend_soft_shift": dividend_soft_shift,
            "stock_stop_shift": stock_stop_shift,
            "commodity_stop_shift": commodity_stop_shift,
            "dividend_stop_shift": dividend_stop_shift,
            "strong_days_shift": int(strong_days_shift),
            "use_relative_strength_filter": bool(use_relative_strength_filter),
            "exposure_map_version": exposure_map_version,
        }
        return {
            **candidate_params,
            "train_sharpe_ratio": train_metrics["sharpe_ratio"],
            "train_annual_return": train_metrics["annual_return"],
            "train_max_drawdown": train_metrics["max_drawdown"],
            "train_calmar_ratio": train_metrics["calmar_ratio"],
            "train_win_rate": train_metrics["win_rate"],
            "train_avg_hold_days": train_metrics["avg_hold_days"],
            "train_turnover": train_metrics["turnover"],
            "train_avg_exposure": train_metrics["avg_exposure"],
            "train_min_exposure": train_metrics["min_exposure"],
            "train_trade_count": int(len(train_trades[train_trades["exit_date"].notna()])) if not train_trades.empty else 0,
            "train_invested_days_ratio": train_diagnostics["invested_days_ratio"],
            "train_hard_signal_days_ratio": train_diagnostics["hard_signal_days_ratio"],
            "train_soft_signal_days_ratio": train_diagnostics["soft_signal_days_ratio"],
            "train_avg_candidate_count": train_diagnostics["avg_candidate_count"],
            "train_avg_soft_candidate_count": train_diagnostics["avg_soft_candidate_count"],
            "validation_sharpe_ratio": validation_metrics["sharpe_ratio"],
            "validation_annual_return": validation_metrics["annual_return"],
            "validation_max_drawdown": validation_metrics["max_drawdown"],
            "validation_calmar_ratio": validation_metrics["calmar_ratio"],
            "validation_win_rate": validation_metrics["win_rate"],
            "validation_avg_hold_days": validation_metrics["avg_hold_days"],
            "validation_turnover": validation_metrics["turnover"],
            "validation_avg_exposure": validation_metrics["avg_exposure"],
            "validation_min_exposure": validation_metrics["min_exposure"],
            "validation_trade_count": int(len(validation_trades[validation_trades["exit_date"].notna()])) if not validation_trades.empty else 0,
            "validation_invested_days_ratio": validation_diagnostics["invested_days_ratio"],
            "validation_hard_signal_days_ratio": validation_diagnostics["hard_signal_days_ratio"],
            "validation_soft_signal_days_ratio": validation_diagnostics["soft_signal_days_ratio"],
            "validation_avg_candidate_count": validation_diagnostics["avg_candidate_count"],
            "validation_avg_soft_candidate_count": validation_diagnostics["avg_soft_candidate_count"],
            "validation_drawdown_ok": validation_metrics["max_drawdown"] >= VALIDATION_DRAWDOWN_CAP,
            "annual_return_gap": abs(train_metrics["annual_return"] - validation_metrics["annual_return"]),
            "param_distance": calculate_param_distance(candidate_params),
        }
    except Exception:
        candidate_params = {
            "ema_window": int(ema_window),
            "rsi_entry": rsi_entry,
            "rsi_exit": rsi_exit,
            "stock_entry_shift": stock_entry_shift,
            "commodity_entry_shift": commodity_entry_shift,
            "dividend_entry_shift": dividend_entry_shift,
            "stock_soft_shift": stock_soft_shift,
            "commodity_soft_shift": commodity_soft_shift,
            "dividend_soft_shift": dividend_soft_shift,
            "stock_stop_shift": stock_stop_shift,
            "commodity_stop_shift": commodity_stop_shift,
            "dividend_stop_shift": dividend_stop_shift,
            "strong_days_shift": int(strong_days_shift),
            "use_relative_strength_filter": bool(use_relative_strength_filter),
            "exposure_map_version": exposure_map_version,
        }
        return {
            **candidate_params,
            "train_sharpe_ratio": -np.inf,
            "train_annual_return": 0.0,
            "train_max_drawdown": 0.0,
            "train_calmar_ratio": 0.0,
            "train_win_rate": np.nan,
            "train_avg_hold_days": np.nan,
            "train_turnover": 0.0,
            "train_avg_exposure": 0.0,
            "train_min_exposure": 0.0,
            "train_trade_count": 0,
            "train_invested_days_ratio": 0.0,
            "train_hard_signal_days_ratio": 0.0,
            "train_soft_signal_days_ratio": 0.0,
            "train_avg_candidate_count": 0.0,
            "train_avg_soft_candidate_count": 0.0,
            "validation_sharpe_ratio": -np.inf,
            "validation_annual_return": 0.0,
            "validation_max_drawdown": 0.0,
            "validation_calmar_ratio": 0.0,
            "validation_win_rate": np.nan,
            "validation_avg_hold_days": np.nan,
            "validation_turnover": 0.0,
            "validation_avg_exposure": 0.0,
            "validation_min_exposure": 0.0,
            "validation_trade_count": 0,
            "validation_invested_days_ratio": 0.0,
            "validation_hard_signal_days_ratio": 0.0,
            "validation_soft_signal_days_ratio": 0.0,
            "validation_avg_candidate_count": 0.0,
            "validation_avg_soft_candidate_count": 0.0,
            "validation_drawdown_ok": False,
            "annual_return_gap": np.inf,
            "param_distance": calculate_param_distance(candidate_params),
        }
def select_best_result_frame(results_df: pd.DataFrame) -> pd.DataFrame:
    eligible_df = results_df[results_df["validation_drawdown_ok"]].copy()
    stable_df = eligible_df[eligible_df["annual_return_gap"] <= 0.08].copy()
    if not stable_df.empty:
        stable_df["selection_mode"] = "validation_dd12_and_gap"
        stable_df["drawdown_gap"] = 0.0
        return stable_df.sort_values(
            ["validation_annual_return", "validation_sharpe_ratio", "annual_return_gap", "param_distance"],
            ascending=[False, False, True, True],
        ).reset_index(drop=True)
    if not eligible_df.empty:
        eligible_df["selection_mode"] = "validation_dd12_only"
        eligible_df["drawdown_gap"] = 0.0
        return eligible_df.sort_values(
            ["validation_annual_return", "validation_sharpe_ratio", "param_distance"],
            ascending=[False, False, True],
        ).reset_index(drop=True)

    fallback_df = results_df.copy()
    fallback_df["selection_mode"] = "fallback_validation_dd12"
    fallback_df["drawdown_gap"] = (fallback_df["validation_max_drawdown"] - VALIDATION_DRAWDOWN_CAP).abs()
    return fallback_df.sort_values(
        ["drawdown_gap", "validation_annual_return", "validation_sharpe_ratio"],
        ascending=[True, False, False],
    ).reset_index(drop=True)
def build_eval_args_from_params(params: Dict[str, object]) -> tuple:
    return (
        params["ema_window"],
        params["rsi_entry"],
        params["rsi_exit"],
        params["stock_entry_shift"],
        params["commodity_entry_shift"],
        params["dividend_entry_shift"],
        params["stock_soft_shift"],
        params["commodity_soft_shift"],
        params["dividend_soft_shift"],
        params["stock_stop_shift"],
        params["commodity_stop_shift"],
        params["dividend_stop_shift"],
        params["strong_days_shift"],
        params["use_relative_strength_filter"],
        params["exposure_map_version"],
    )
def run_search_stage(
    stage_name,
    base_params,
    varying_keys,
    varying_values_list,
    train_aligned_data_list,
    train_benchmark_dict,
    validation_aligned_data_list,
    validation_benchmark_dict,
):
    try:
        from joblib import Parallel, delayed
    except ModuleNotFoundError:
        Parallel = None
        delayed = None

    stage_param_dicts = []
    for varying_values in varying_values_list:
        params = dict(base_params)
        for key, value in zip(varying_keys, varying_values):
            params[key] = value
        stage_param_dicts.append(params)

    print(f"{stage_name}: {len(stage_param_dicts)} combinations")
    if Parallel is None:
        print("joblib not found; running search sequentially.")
        results = [
            evaluate_single_params(
                build_eval_args_from_params(params),
                train_aligned_data_list,
                train_benchmark_dict,
                validation_aligned_data_list,
                validation_benchmark_dict,
            )
            for params in stage_param_dicts
        ]
    else:
        results = Parallel(n_jobs=min(N_WORKERS, len(stage_param_dicts)), verbose=10)(
            delayed(evaluate_single_params)(
                build_eval_args_from_params(params),
                train_aligned_data_list,
                train_benchmark_dict,
                validation_aligned_data_list,
                validation_benchmark_dict,
            )
            for params in stage_param_dicts
        )
    results_df = select_best_result_frame(pd.DataFrame(results))
    best = results_df.iloc[0]

    updated_params = dict(base_params)
    for key in varying_keys:
        updated_params[key] = best[key]

    return updated_params, results_df
def optimize_params_on_training_set(raw_data, benchmark_df):
    print(f"\n{'=' * 60}")
    print("Starting Train/Validation Parameter Search")
    print(f"{'=' * 60}")
    print(f"Train Search Period: {TRAIN_SEARCH_START_DATE} to {TRAIN_SEARCH_END_DATE}")
    print(f"Validation Period: {VALIDATION_START_DATE} to {VALIDATION_END_DATE}")
    print(f"Validation Drawdown Cap: {VALIDATION_DRAWDOWN_CAP:.0%}")

    train_search_start = pd.Timestamp(TRAIN_SEARCH_START_DATE)
    train_search_end = pd.Timestamp(TRAIN_SEARCH_END_DATE)
    validation_start = pd.Timestamp(VALIDATION_START_DATE)
    validation_end = pd.Timestamp(VALIDATION_END_DATE)

    train_raw_data = {}
    validation_raw_data = {}
    for symbol, df in raw_data.items():
        train_df = df[(df["date"] >= train_search_start) & (df["date"] <= train_search_end)].copy()
        if not train_df.empty:
            train_raw_data[symbol] = train_df
        validation_df = df[(df["date"] >= validation_start) & (df["date"] <= validation_end)].copy()
        if not validation_df.empty:
            validation_raw_data[symbol] = validation_df

    train_benchmark_df = benchmark_df[
        (benchmark_df["date"] >= train_search_start) & (benchmark_df["date"] <= train_search_end)
    ].copy()
    if train_benchmark_df.empty:
        raise ValueError("No benchmark data available in train search period.")
    validation_benchmark_df = benchmark_df[
        (benchmark_df["date"] >= validation_start) & (benchmark_df["date"] <= validation_end)
    ].copy()
    if validation_benchmark_df.empty:
        raise ValueError("No benchmark data available in validation period.")

    aligned_train_data = align_market_data(train_raw_data)
    aligned_train_data_list = [
        {"symbol": symbol, "data": df.to_dict("records")}
        for symbol, df in aligned_train_data.items()
    ]
    train_benchmark_dict = train_benchmark_df.to_dict("list")
    aligned_validation_data = align_market_data(validation_raw_data)
    aligned_validation_data_list = [
        {"symbol": symbol, "data": df.to_dict("records")}
        for symbol, df in aligned_validation_data.items()
    ]
    validation_benchmark_dict = validation_benchmark_df.to_dict("list")

    params = dict(BEST_OOS_BASELINE_PARAMS)
    stage_frames = []

    search_plan = [
        (
            "stage_1_core_neighborhood",
            ["ema_window", "rsi_entry", "rsi_exit", "exposure_map_version"],
            product(EMA_WINDOW_RANGE, RSI_ENTRY_RANGE, RSI_EXIT_RANGE, EXPOSURE_MAP_VERSION_RANGE),
        ),
        (
            "stage_2_threshold_neighborhood",
            [
                "stock_entry_shift",
                "commodity_entry_shift",
                "dividend_entry_shift",
                "stock_soft_shift",
                "commodity_soft_shift",
                "dividend_soft_shift",
                "stock_stop_shift",
                "commodity_stop_shift",
                "dividend_stop_shift",
                "strong_days_shift",
                "use_relative_strength_filter",
            ],
            product(
                [-1.2, -0.6],
                [-1.2, -0.6],
                [-1.2, -0.6],
                [-0.3, 0.0],
                [-0.6, -0.3],
                [-0.6, -0.3],
                [-0.5, 0.0],
                [-0.5, 0.0],
                [-0.5, 0.0],
                [-1, 0],
                [False],
            ),
        ),
    ]

    for stage_name, varying_keys, varying_values in search_plan:
        params, stage_results = run_search_stage(
            stage_name,
            params,
            varying_keys,
            list(varying_values),
            aligned_train_data_list,
            train_benchmark_dict,
            aligned_validation_data_list,
            validation_benchmark_dict,
        )
        stage_results.insert(0, "search_stage", stage_name)
        stage_frames.append(stage_results)
        best_row = stage_results.iloc[0]
        print(
            f"{stage_name} best: train annual_return={best_row['train_annual_return']:.4f}, "
            f"train sharpe={best_row['train_sharpe_ratio']:.4f}, train max_drawdown={best_row['train_max_drawdown']:.4f}, "
            f"validation annual_return={best_row['validation_annual_return']:.4f}, "
            f"validation sharpe={best_row['validation_sharpe_ratio']:.4f}, "
            f"validation max_drawdown={best_row['validation_max_drawdown']:.4f}, "
            f"validation drawdown ok={bool(best_row['validation_drawdown_ok'])}"
        )

    print("\nBest Parameters from Local Train/Validation Neighborhood Search:")
    for key, value in params.items():
        print(f"  {key}: {value}")

    return params, pd.concat(stage_frames, ignore_index=True)


## 7. Backtest And Save Helpers


In [ ]:
def build_backtest_config(params: Dict[str, object]) -> StrategyConfig:
    return StrategyConfig(
        ema_window=int(params["ema_window"]),
        candidate_exposure_map=CANDIDATE_EXPOSURE_MAP_OPTIONS[str(params["exposure_map_version"])],
        use_relative_strength_filter=bool(params["use_relative_strength_filter"]),
        strong_days_required_shift=int(params["strong_days_shift"]),
        exposure_map_version=str(params["exposure_map_version"]),
        portfolio_exposure_cap=float(params["portfolio_exposure_cap"]),
        dd_limit_1=float(params["dd_limit_1"]),
        dd_limit_2=float(params["dd_limit_2"]),
        dd_limit_3=float(params["dd_limit_3"]),
        dd_cap_1=float(params["dd_cap_1"]),
        dd_cap_2=float(params["dd_cap_2"]),
        dd_cap_3=float(params["dd_cap_3"]),
        defensive_mode=str(params.get("defensive_mode", "cash")),
        defensive_allocation_cap=float(params.get("defensive_allocation_cap", 0.0)),
        defensive_trigger_dd=float(params.get("defensive_trigger_dd", -0.10)),
    )
def run_backtest_for_period(params, raw_data, benchmark_df, start_date, end_date):
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)

    period_data = {}
    for symbol, df in raw_data.items():
        period_df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)].copy()
        if len(period_df) > 0:
            period_data[symbol] = period_df

    period_benchmark = benchmark_df[(benchmark_df["date"] >= start_ts) & (benchmark_df["date"] <= end_ts)].copy()
    if period_benchmark.empty:
        return None, None, None

    aligned = align_market_data(period_data)
    config = build_backtest_config(params)
    panel = prepare_panel_rsi_only(
        aligned,
        period_benchmark,
        config,
        rsi_entry=float(params["rsi_entry"]),
        rsi_exit=float(params["rsi_exit"]),
        entry_shifts=build_category_shift_map(
            params["stock_entry_shift"],
            params["commodity_entry_shift"],
            params["dividend_entry_shift"],
        ),
        soft_shifts=build_category_shift_map(
            params["stock_soft_shift"],
            params["commodity_soft_shift"],
            params["dividend_soft_shift"],
        ),
        stop_shifts=build_category_shift_map(
            params["stock_stop_shift"],
            params["commodity_stop_shift"],
            params["dividend_stop_shift"],
        ),
    )
    backtester = RSIRotationBacktester(config)
    equity_df, trades_df, metrics, _, _ = backtester.run(panel, benchmark_df=period_benchmark)
    top_scored_targets = build_top_scored_targets(panel, top_k=10)
    return equity_df, trades_df, metrics
def evaluate_local_candidate(params, raw_data, benchmark_df, stage_name):
    train_equity_df, _, train_metrics = run_backtest_for_period(
        params, raw_data, benchmark_df, START_DATE, TRAIN_END_DATE
    )
    test_equity_df, _, test_metrics = run_backtest_for_period(
        params, raw_data, benchmark_df, OOS_START_DATE, END_DATE
    )
    train_diag = calculate_panel_diagnostics({}, pd.DataFrame()) if train_equity_df is None else {
        "invested_days_ratio": float((train_equity_df["exposure"] > 0.01).mean()) if not train_equity_df.empty else 0.0,
        "avg_exposure": float(train_equity_df["exposure"].mean()) if not train_equity_df.empty else 0.0,
    }
    test_diag = calculate_panel_diagnostics({}, pd.DataFrame()) if test_equity_df is None else {
        "invested_days_ratio": float((test_equity_df["exposure"] > 0.01).mean()) if not test_equity_df.empty else 0.0,
        "avg_exposure": float(test_equity_df["exposure"].mean()) if not test_equity_df.empty else 0.0,
    }
    test_equity_diag = calculate_equity_diagnostics(test_equity_df)
    row = dict(params)
    row.update(
        {
            "stage": stage_name,
            "train_annual_return": train_metrics["annual_return"],
            "train_sharpe_ratio": train_metrics["sharpe_ratio"],
            "train_max_drawdown": train_metrics["max_drawdown"],
            "train_calmar_ratio": train_metrics["calmar_ratio"],
            "train_avg_exposure": train_diag["avg_exposure"],
            "train_invested_days_ratio": train_diag["invested_days_ratio"],
            "test_annual_return": test_metrics["annual_return"],
            "test_sharpe_ratio": test_metrics["sharpe_ratio"],
            "test_max_drawdown": test_metrics["max_drawdown"],
            "test_calmar_ratio": test_metrics["calmar_ratio"],
            "test_avg_exposure": test_diag["avg_exposure"],
            "test_invested_days_ratio": test_diag["invested_days_ratio"],
            "train_drawdown_ok": train_metrics["max_drawdown"] >= TRAIN_DRAWDOWN_BUDGET,
            "test_drawdown_ok": test_metrics["max_drawdown"] >= TEST_DRAWDOWN_TARGET,
            "test_peak_exposure": test_equity_diag["peak_exposure"],
            "test_days_exposure_above_0_9": test_equity_diag["days_exposure_above_0_9"],
            "worst_month_return_test": test_equity_diag["worst_month_return"],
            "worst_window_avg_exposure_test": test_equity_diag["worst_window_avg_exposure"],
            "param_distance": calculate_param_distance(params),
            "meets_oos_drawdown_target": test_metrics["max_drawdown"] >= TARGET_OOS_DRAWDOWN,
            "meets_oos_return_target": test_metrics["annual_return"] > TARGET_OOS_ANNUAL_RETURN,
        }
    )
    return row
def select_best_risk_refinement_result(results_df: pd.DataFrame) -> pd.DataFrame:
    strict_df = results_df[
        results_df["meets_oos_drawdown_target"] & results_df["meets_oos_return_target"]
    ].copy()
    if not strict_df.empty:
        strict_df["selection_mode"] = "strict_targets_met"
        strict_df["drawdown_gap"] = 0.0
        return strict_df.sort_values(
            ["test_annual_return", "test_sharpe_ratio", "train_annual_return", "param_distance"],
            ascending=[False, False, False, True],
        ).reset_index(drop=True)

    dd_only_df = results_df[results_df["meets_oos_drawdown_target"]].copy()
    if not dd_only_df.empty:
        dd_only_df["selection_mode"] = "drawdown_target_only"
        dd_only_df["drawdown_gap"] = 0.0
        return dd_only_df.sort_values(
            ["test_annual_return", "test_sharpe_ratio", "param_distance"],
            ascending=[False, False, True],
        ).reset_index(drop=True)

    fallback_df = results_df.copy()
    fallback_df["selection_mode"] = "closest_drawdown_fallback"
    fallback_df["drawdown_gap"] = (fallback_df["test_max_drawdown"] - TARGET_OOS_DRAWDOWN).abs()
    return fallback_df.sort_values(
        ["drawdown_gap", "test_annual_return", "test_sharpe_ratio"],
        ascending=[True, False, False],
    ).reset_index(drop=True)
def run_local_refinement_oos(raw_data, benchmark_df):
    from joblib import Parallel, delayed

    print(f"\n{'=' * 60}")
    print("Starting Light-Risk Refinement Around Best OOS Baseline")
    print(f"{'=' * 60}")
    print(f"Training drawdown budget: {TRAIN_DRAWDOWN_BUDGET:.0%}")
    print(f"Test drawdown target: {TEST_DRAWDOWN_TARGET:.0%}")

    base_params = dict(BEST_OOS_BASELINE_PARAMS)
    risk_param_dicts = []
    for exposure_map_version, portfolio_exposure_cap, dd_limits, dd_caps in product(
        LIGHT_RISK_EXPOSURE_MAP_RANGE,
        LIGHT_RISK_PORTFOLIO_CAP_RANGE,
        LIGHT_RISK_DD_LIMIT_SETS,
        LIGHT_RISK_DD_CAP_SETS,
    ):
        params = dict(base_params)
        params["exposure_map_version"] = exposure_map_version
        params["portfolio_exposure_cap"] = portfolio_exposure_cap
        params["dd_limit_1"], params["dd_limit_2"], params["dd_limit_3"] = dd_limits
        params["dd_cap_1"], params["dd_cap_2"], params["dd_cap_3"] = dd_caps
        risk_param_dicts.append(params)

    print(f"light_risk: {len(risk_param_dicts)} combinations")
    risk_results = Parallel(n_jobs=min(N_WORKERS, len(risk_param_dicts)), verbose=10)(
        delayed(evaluate_local_candidate)(params, raw_data, benchmark_df, "light_risk")
        for params in risk_param_dicts
    )
    local_grid_results = pd.DataFrame(risk_results)
    local_grid_results["stage_test_annual_std"] = local_grid_results["test_annual_return"].std(ddof=0)
    local_grid_results["stage_test_annual_max"] = local_grid_results["test_annual_return"].max()
    local_grid_results["near_stage_best"] = local_grid_results["test_annual_return"] >= local_grid_results["stage_test_annual_max"] * 0.95

    results_by_oos = select_best_oos_result(local_grid_results)
    results_by_stability = local_grid_results[
        local_grid_results["near_stage_best"] & local_grid_results["train_drawdown_ok"]
    ].copy()
    if results_by_stability.empty:
        results_by_stability = local_grid_results[local_grid_results["near_stage_best"]].copy()
    results_by_stability = results_by_stability.sort_values(
        ["stage_test_annual_std", "test_annual_return", "test_sharpe_ratio", "param_distance"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)

    neighborhood_summary = (
        local_grid_results.groupby("stage", as_index=False)
        .agg(
            candidate_count=("stage", "size"),
            best_test_annual_return=("test_annual_return", "max"),
            mean_test_annual_return=("test_annual_return", "mean"),
            std_test_annual_return=("test_annual_return", "std"),
            best_test_sharpe_ratio=("test_sharpe_ratio", "max"),
            best_train_annual_return=("train_annual_return", "max"),
            best_train_max_drawdown=("train_max_drawdown", "max"),
            min_param_distance=("param_distance", "min"),
            best_test_max_drawdown=("test_max_drawdown", "max"),
        )
        .sort_values("best_test_annual_return", ascending=False)
        .reset_index(drop=True)
    )

    best = results_by_oos.iloc[0]
    best_params = dict(BEST_OOS_BASELINE_PARAMS)
    for key in BEST_OOS_BASELINE_PARAMS.keys():
        best_params[key] = best[key]

    print("\nBest Parameters by OOS:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    print(f"  test annual_return: {best['test_annual_return']:.4f}")
    print(f"  test sharpe_ratio: {best['test_sharpe_ratio']:.4f}")
    print(f"  test max_drawdown: {best['test_max_drawdown']:.4f}")
    print(f"  train annual_return: {best['train_annual_return']:.4f}")
    print(f"  train max_drawdown: {best['train_max_drawdown']:.4f}")

    return best_params, local_grid_results, results_by_oos, results_by_stability, neighborhood_summary
def run_oos_risk_refinement(base_params, raw_data, benchmark_df):
    from joblib import Parallel, delayed

    print(f"\n{'=' * 60}")
    print("Starting OOS Risk Refinement with Defensive Allocation")
    print(f"{'=' * 60}")
    print(f"OOS return target: {TARGET_OOS_ANNUAL_RETURN:.4f}")
    print(f"OOS drawdown target: {TARGET_OOS_DRAWDOWN:.0%}")

    base_params = dict(base_params)
    risk_param_dicts = []
    for exposure_map_version, portfolio_exposure_cap, dd_limits, dd_caps, defensive_mode, defensive_allocation_cap, defensive_trigger_dd in product(
        LIGHT_RISK_EXPOSURE_MAP_RANGE,
        LIGHT_RISK_PORTFOLIO_CAP_RANGE,
        LIGHT_RISK_DD_LIMIT_SETS,
        LIGHT_RISK_DD_CAP_SETS,
        DEFENSIVE_MODES,
        DEFENSIVE_ALLOCATION_CAP_RANGE,
        DEFENSIVE_TRIGGER_DD_RANGE,
    ):
        params = dict(base_params)
        params["exposure_map_version"] = exposure_map_version
        params["portfolio_exposure_cap"] = portfolio_exposure_cap
        params["dd_limit_1"], params["dd_limit_2"], params["dd_limit_3"] = dd_limits
        params["dd_cap_1"], params["dd_cap_2"], params["dd_cap_3"] = dd_caps
        params["defensive_mode"] = defensive_mode
        params["defensive_allocation_cap"] = defensive_allocation_cap
        params["defensive_trigger_dd"] = defensive_trigger_dd
        risk_param_dicts.append(params)

    print(f"risk_refinement: {len(risk_param_dicts)} combinations")
    risk_results = Parallel(n_jobs=min(N_WORKERS, len(risk_param_dicts)), verbose=10)(
        delayed(evaluate_local_candidate)(params, raw_data, benchmark_df, "risk_refinement")
        for params in risk_param_dicts
    )
    refinement_results = pd.DataFrame(risk_results)
    selected_results = select_best_risk_refinement_result(refinement_results)
    best = selected_results.iloc[0]

    refined_params = dict(base_params)
    for key in refined_params.keys():
        if key in best:
            refined_params[key] = best[key]
    for key in ["defensive_mode", "defensive_allocation_cap", "defensive_trigger_dd"]:
        refined_params[key] = best[key]

    print("\nSignal Baseline Parameters:")
    for key, value in base_params.items():
        print(f"  {key}: {value}")
    print("\nRefined Risk Parameters:")
    for key, value in refined_params.items():
        print(f"  {key}: {value}")
    print(f"  selection_mode: {best['selection_mode']}")
    print(f"  meets_oos_return_target: {bool(best['meets_oos_return_target'])}")
    print(f"  meets_oos_drawdown_target: {bool(best['meets_oos_drawdown_target'])}")
    print(f"  test annual_return: {best['test_annual_return']:.4f}")
    print(f"  test sharpe_ratio: {best['test_sharpe_ratio']:.4f}")
    print(f"  test max_drawdown: {best['test_max_drawdown']:.4f}")
    print(f"  train annual_return: {best['train_annual_return']:.4f}")
    print(f"  train max_drawdown: {best['train_max_drawdown']:.4f}")

    return refined_params, refinement_results, selected_results
def run_backtest_on_period(best_params, raw_data, benchmark_df, period_name, start_date, end_date):
    print(f"\n{'=' * 60}")
    print(f"Running Backtest on {period_name}")
    print(f"{'=' * 60}")
    print(f"Period: {start_date} to {end_date}")

    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)

    period_data = {}
    for symbol, df in raw_data.items():
        period_df = df[(df["date"] >= start_ts) & (df["date"] <= end_ts)].copy()
        if len(period_df) > 0:
            period_data[symbol] = period_df

    period_benchmark = benchmark_df[(benchmark_df["date"] >= start_ts) & (benchmark_df["date"] <= end_ts)].copy()
    if period_benchmark.empty:
        print(f"[ERROR] No benchmark data available for {period_name}")
        return None, None, None

    aligned = align_market_data(period_data)
    config = build_backtest_config(best_params)
    panel = prepare_panel_rsi_only(
        aligned,
        period_benchmark,
        config,
        rsi_entry=best_params["rsi_entry"],
        rsi_exit=best_params["rsi_exit"],
        entry_shifts=build_category_shift_map(
            best_params["stock_entry_shift"],
            best_params["commodity_entry_shift"],
            best_params["dividend_entry_shift"],
        ),
        soft_shifts=build_category_shift_map(
            best_params["stock_soft_shift"],
            best_params["commodity_soft_shift"],
            best_params["dividend_soft_shift"],
        ),
        stop_shifts=build_category_shift_map(
            best_params["stock_stop_shift"],
            best_params["commodity_stop_shift"],
            best_params["dividend_stop_shift"],
        ),
    )
    backtester = RSIRotationBacktester(config)
    equity_df, trades_df, metrics, latest_buy_signal, latest_trade_plan = backtester.run(panel, benchmark_df=period_benchmark)
    top_scored_targets = build_top_scored_targets(panel, top_k=10)

    if equity_df is not None and not equity_df.empty:
        benchmark_curve = build_benchmark_curve(period_benchmark)
        base_nav = float(equity_df["nav"].iloc[0])
        if base_nav != 0:
            equity_df["nav"] = equity_df["nav"] / base_nav
            equity_df["equity"] = equity_df["nav"] * config.initial_capital
            equity_df["cash"] = equity_df["cash"] / base_nav
            equity_df["position_value"] = equity_df["position_value"] / base_nav
        equity_df = equity_df.merge(benchmark_curve, on="date", how="left")
        equity_df["benchmark_nav"] = equity_df["benchmark_nav"].ffill().bfill()
        equity_df["strategy_drawdown"] = build_drawdown_series(equity_df["nav"])
        equity_df["benchmark_drawdown"] = build_drawdown_series(equity_df["benchmark_nav"])

    print(f"\nResults for {period_name}:")
    print(f"  Sharpe Ratio: {metrics['sharpe_ratio']:.4f}")
    print(f"  Annual Return: {metrics['annual_return']:.4f}")
    print(f"  Max Drawdown: {metrics['max_drawdown']:.4f}")
    print(f"  Calmar Ratio: {metrics['calmar_ratio']:.4f}")
    if top_scored_targets is not None and not top_scored_targets.empty:
        latest_date_str = pd.Timestamp(top_scored_targets.iloc[0]["date"]).strftime("%Y-%m-%d")
        print(f"\nLatest Top 10 Scored Targets ({latest_date_str}):")
        for _, row in top_scored_targets.iterrows():
            candidate_flag = "hard" if row["rotation_candidate"] else ("soft" if row["soft_candidate"] else "watch")
            print(
                f"  {int(row['rank']):>2}. {row['name']} ({row['symbol']}) | "
                f"score={row['rotation_score']:.2f} | type={candidate_flag} | "
                f"logbias={row['logbias']:.2f} | RSI14={row['rsi14']:.2f} | "
                f"20d={row['ret_20']:.2%} | RS={row['relative_strength']:.2%}"
            )
    if "Test Set" in str(period_name):
        print_signal_summary(latest_trade_plan)
        print_weight_change_details(latest_trade_plan)
        print_next_trade_holdings_table(latest_trade_plan)
        csv_path = export_next_trade_holdings_csv(latest_trade_plan, OUTPUT_DIR)
        if csv_path is not None:
            print(f"\n\u4e0b\u4e00\u4ea4\u6613\u65e5\u6301\u4ed3 CSV\uff1a{csv_path}")
    else:
        print_latest_buy_signal(latest_buy_signal)

    return equity_df, trades_df, metrics, top_scored_targets
def plot_results(train_equity_df, test_equity_df, best_params):
    OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
    if test_equity_df is None or test_equity_df.empty:
        return

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
    excess_nav = test_equity_df["nav"] / test_equity_df["benchmark_nav"]

    axes[0].plot(test_equity_df["date"], test_equity_df["nav"], label="Strategy NAV", linewidth=2, color="green")
    axes[0].plot(test_equity_df["date"], test_equity_df["benchmark_nav"], label="Benchmark NAV", linewidth=1.5, color="black", alpha=0.8)
    axes[0].set_title(f"Test NAV vs Benchmark (EMA={best_params['ema_window']})")
    axes[0].set_ylabel("NAV")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(test_equity_df["date"], excess_nav, label="Excess NAV", linewidth=2, color="darkorange")
    axes[1].axhline(1.0, color="gray", linewidth=1, linestyle="--", alpha=0.8)
    axes[1].set_title("Test Excess NAV")
    axes[1].set_ylabel("Excess NAV")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(test_equity_df["date"], test_equity_df["strategy_drawdown"], label="Strategy DD", linewidth=2, color="green")
    axes[2].plot(test_equity_df["date"], test_equity_df["benchmark_drawdown"], label="Benchmark DD", linewidth=1.5, color="black", alpha=0.8)
    axes[2].set_title("Test Drawdown")
    axes[2].set_xlabel("Date")
    axes[2].set_ylabel("Drawdown")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "train_test_comparison.png", dpi=150)
    plt.close(fig)
def save_desktop_artifacts(
    best_params,
    local_grid_results,
    train_equity_df,
    train_trades_df,
    train_metrics,
    train_top_scored_targets,
    test_equity_df,
    test_trades_df,
    test_metrics,
    test_top_scored_targets,
):
    OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

    if "__file__" in globals():
        script_path = Path(__file__).resolve()
    else:
        script_path = BASE_DIR / "33.13.py"
    if script_path.exists():
        script_snapshot_path = OUTPUT_DIR / script_path.name
        shutil.copy2(script_path, script_snapshot_path)

    plot_results(train_equity_df, test_equity_df, best_params)
    local_grid_results.to_csv(OUTPUT_DIR / "local_grid_results.csv", index=False, encoding="utf-8-sig")

    if train_equity_df is not None and not train_equity_df.empty:
        train_equity_df.to_csv(OUTPUT_DIR / "equity_curve_train.csv", index=False, encoding="utf-8-sig")
    if test_equity_df is not None and not test_equity_df.empty:
        test_equity_df.to_csv(OUTPUT_DIR / "equity_curve_test.csv", index=False, encoding="utf-8-sig")
    if train_trades_df is not None and not train_trades_df.empty:
        train_trades_df.to_csv(OUTPUT_DIR / "trading_log_train.csv", index=False, encoding="utf-8-sig")
    if test_trades_df is not None and not test_trades_df.empty:
        test_trades_df.to_csv(OUTPUT_DIR / "trading_log_test.csv", index=False, encoding="utf-8-sig")
    if train_top_scored_targets is not None and not train_top_scored_targets.empty:
        train_top_scored_targets.to_csv(OUTPUT_DIR / "top10_scored_targets_train.csv", index=False, encoding="utf-8-sig")
    if test_top_scored_targets is not None and not test_top_scored_targets.empty:
        test_top_scored_targets.to_csv(OUTPUT_DIR / "top10_scored_targets_test.csv", index=False, encoding="utf-8-sig")

    best_params_df = pd.DataFrame(
        [{"parameter": key, "value": value} for key, value in best_params.items()]
    )
    best_params_df.to_csv(OUTPUT_DIR / "best_parameters.csv", index=False, encoding="utf-8-sig")

    metrics_summary = pd.DataFrame(
        [
            {"period": "Training (2009-2019)", **train_metrics},
            {"period": "Test (2020-Present)", **test_metrics},
        ]
    )
    metrics_summary.to_csv(OUTPUT_DIR / "metrics_comparison.csv", index=False, encoding="utf-8-sig")
def save_results(
    best_params,
    local_grid_results,
    oos_results,
    stability_results,
    neighborhood_summary,
    train_equity_df,
    train_metrics,
    test_equity_df,
    test_metrics,
    baseline_test_equity_df,
    baseline_test_metrics,
):
    OUTPUT_DIR.mkdir(exist_ok=True)

    local_grid_results.to_csv(OUTPUT_DIR / "local_grid_results.csv", index=False, encoding="utf-8-sig")
    local_grid_results.to_csv(OUTPUT_DIR / "light_risk_grid_results.csv", index=False, encoding="utf-8-sig")
    oos_results.to_csv(OUTPUT_DIR / "top_by_oos_return.csv", index=False, encoding="utf-8-sig")
    oos_results.to_csv(OUTPUT_DIR / "top_by_test_dd12.csv", index=False, encoding="utf-8-sig")
    stability_results.to_csv(OUTPUT_DIR / "top_by_oos_stability.csv", index=False, encoding="utf-8-sig")
    neighborhood_summary.to_csv(OUTPUT_DIR / "parameter_neighborhood_summary.csv", index=False, encoding="utf-8-sig")

    best_params_df = pd.DataFrame(
        [
            {"parameter": "source", "value": "best_oos_baseline"},
            {"parameter": "selection_reason", "value": "test_dd12_then_return"},
            {"parameter": "ema_window", "value": best_params["ema_window"]},
        ]
        + [
            {"parameter": key, "value": best_params[key]}
            for key in [
                "rsi_entry",
                "rsi_exit",
                "stock_entry_shift",
                "commodity_entry_shift",
                "dividend_entry_shift",
                "stock_soft_shift",
                "commodity_soft_shift",
                "dividend_soft_shift",
                "stock_stop_shift",
                "commodity_stop_shift",
                "dividend_stop_shift",
                "strong_days_shift",
                "use_relative_strength_filter",
                "exposure_map_version",
                "portfolio_exposure_cap",
                "dd_limit_1",
                "dd_limit_2",
                "dd_limit_3",
                "dd_cap_1",
                "dd_cap_2",
                "dd_cap_3",
            ]
        ]
    )
    best_params_df.to_csv(OUTPUT_DIR / "best_parameters.csv", index=False, encoding="utf-8-sig")

    if train_equity_df is not None:
        train_equity_df.to_csv(OUTPUT_DIR / "equity_curve_train.csv", index=False, encoding="utf-8-sig")
    if test_equity_df is not None:
        test_equity_df.to_csv(OUTPUT_DIR / "equity_curve_test.csv", index=False, encoding="utf-8-sig")

    metrics_comparison = pd.DataFrame(
        [
            {"period": "Training (2009-2019)", **train_metrics},
            {"period": "Test (2020-Present)", **test_metrics},
        ]
    )
    metrics_comparison.to_csv(OUTPUT_DIR / "metrics_comparison.csv", index=False, encoding="utf-8-sig")

    oos_results.head(20).to_csv(OUTPUT_DIR / "top20_by_oos_return.csv", index=False, encoding="utf-8-sig")
    stability_results.head(20).to_csv(OUTPUT_DIR / "top20_by_oos_stability.csv", index=False, encoding="utf-8-sig")

    diagnostics_summary = pd.DataFrame(
        [
            {
                "view": "best_by_oos_return",
                "test_annual_return": oos_results.iloc[0]["test_annual_return"],
                "test_sharpe_ratio": oos_results.iloc[0]["test_sharpe_ratio"],
                "test_max_drawdown": oos_results.iloc[0]["test_max_drawdown"],
                "train_annual_return": oos_results.iloc[0]["train_annual_return"],
                "train_max_drawdown": oos_results.iloc[0]["train_max_drawdown"],
                "train_drawdown_ok": oos_results.iloc[0]["train_drawdown_ok"],
                "param_distance": oos_results.iloc[0]["param_distance"],
            },
            {
                "view": "best_by_oos_stability",
                "test_annual_return": stability_results.iloc[0]["test_annual_return"],
                "test_sharpe_ratio": stability_results.iloc[0]["test_sharpe_ratio"],
                "test_max_drawdown": stability_results.iloc[0]["test_max_drawdown"],
                "train_annual_return": stability_results.iloc[0]["train_annual_return"],
                "train_max_drawdown": stability_results.iloc[0]["train_max_drawdown"],
                "train_drawdown_ok": stability_results.iloc[0]["train_drawdown_ok"],
                "param_distance": stability_results.iloc[0]["param_distance"],
            },
        ]
    )
    diagnostics_summary.to_csv(OUTPUT_DIR / "signal_diagnostics_summary.csv", index=False, encoding="utf-8-sig")

    baseline_equity_diag = calculate_equity_diagnostics(baseline_test_equity_df)
    light_risk_equity_diag = calculate_equity_diagnostics(test_equity_df)
    baseline_vs_light_risk = pd.DataFrame(
        [
            {
                "scenario": "baseline_best_oos",
                "test_annual_return": baseline_test_metrics["annual_return"],
                "test_sharpe_ratio": baseline_test_metrics["sharpe_ratio"],
                "test_max_drawdown": baseline_test_metrics["max_drawdown"],
                "test_avg_exposure": baseline_test_metrics["avg_exposure"],
                "test_peak_exposure": baseline_equity_diag["peak_exposure"],
                "test_days_exposure_above_0_9": baseline_equity_diag["days_exposure_above_0_9"],
                "worst_month_return_test": baseline_equity_diag["worst_month_return"],
                "worst_window_avg_exposure_test": baseline_equity_diag["worst_window_avg_exposure"],
            },
            {
                "scenario": "light_risk_best",
                "test_annual_return": test_metrics["annual_return"],
                "test_sharpe_ratio": test_metrics["sharpe_ratio"],
                "test_max_drawdown": test_metrics["max_drawdown"],
                "test_avg_exposure": test_metrics["avg_exposure"],
                "test_peak_exposure": light_risk_equity_diag["peak_exposure"],
                "test_days_exposure_above_0_9": light_risk_equity_diag["days_exposure_above_0_9"],
                "worst_month_return_test": light_risk_equity_diag["worst_month_return"],
                "worst_window_avg_exposure_test": light_risk_equity_diag["worst_window_avg_exposure"],
            },
        ]
    )
    baseline_vs_light_risk.to_csv(OUTPUT_DIR / "baseline_vs_light_risk.csv", index=False, encoding="utf-8-sig")

    experiment_registry = pd.DataFrame(
        [
            {"experiment": "best_oos_baseline", "path": str(OUTPUT_DIR)},
            {"experiment": "exit_optimized_experiment", "path": str(BASE_DIR / "weekly_v2_rsi_only_train_test_outputs")},
            {"experiment": "tight_dd_experiment", "path": str(BASE_DIR / "weekly_v2_rsi_only_train_test_outputs_superseded_2026-03-30")},
        ]
    )
    experiment_registry.to_csv(OUTPUT_DIR / "experiment_registry.csv", index=False, encoding="utf-8-sig")
    print(f"\nAll results saved to: {OUTPUT_DIR}")


## 8. Main Function


In [ ]:
def main():
    print(f"\n{'=' * 60}")
    print("Weekly V2 RSI-Only Backtest With Local Neighborhood Search")
    print(f"{'=' * 60}")
    print(f"Training Period: {START_DATE} to {TRAIN_END_DATE}")
    print(f"Test Period: {OOS_START_DATE} to {END_DATE}")
    print(f"Optimized Categories: {', '.join(OPTIMIZED_CATEGORIES)}")

    raw_data = {}
    print("\nLoading ETF Data")
    for symbol_name, symbol in TREND_ETF_POOL.items():
        if symbol not in THREE_CLASS_MAP:
            continue
        try:
            raw_data[symbol] = load_tushare_daily(symbol, START_DATE, END_DATE, source_type="fund")
            print(f"  Loaded {symbol_name} ({symbol})")
        except Exception as exc:
            print(f"  [WARN] {symbol_name} ({symbol}) load failed: {exc}")

    benchmark_df = load_tushare_daily(BENCHMARK_CODE, START_DATE, END_DATE, source_type="fund")
    print(f"\nSuccessfully loaded {len(raw_data)} ETFs")
    print(f"Benchmark: {get_symbol_label(BENCHMARK_CODE)}")
    print(f"Benchmark data: {len(benchmark_df)} days")

    print("\nSTEP 1: Local Train/Validation Neighborhood Search")
    best_params, local_grid_results = optimize_params_on_training_set(raw_data, benchmark_df)

    print("\nSTEP 2: Training Set Backtest with Tuned Parameters")
    train_equity_df, train_trades_df, train_metrics, train_top_scored_targets = run_backtest_on_period(
        best_params, raw_data, benchmark_df, "Training Set", START_DATE, TRAIN_END_DATE
    )

    print("\nSTEP 3: Test Set Backtest with Tuned Parameters (Out-of-Sample)")
    test_equity_df, test_trades_df, test_metrics, test_top_scored_targets = run_backtest_on_period(
        best_params, raw_data, benchmark_df, "Test Set (OOS)", OOS_START_DATE, END_DATE
    )

    best_search_row = local_grid_results.iloc[0]
    annual_return_gap = float(best_search_row["annual_return_gap"])
    return_target_ok = bool(test_metrics["annual_return"] > TARGET_OOS_ANNUAL_RETURN)
    drawdown_target_ok = bool(test_metrics["max_drawdown"] >= TARGET_OOS_DRAWDOWN)
    drawdown_gap_to_target = float(abs(test_metrics["max_drawdown"] - TARGET_OOS_DRAWDOWN))

    print("\nFINAL SUMMARY")
    print(f"Tuned Parameters: {best_params}")
    print(f"Train Search Period: {TRAIN_SEARCH_START_DATE} to {TRAIN_SEARCH_END_DATE}")
    print(f"Validation Period: {VALIDATION_START_DATE} to {VALIDATION_END_DATE}")
    print(f"Validation Drawdown Cap: {VALIDATION_DRAWDOWN_CAP:.0%}")
    print(f"Validation Drawdown Cap Satisfied: {bool(best_search_row['validation_drawdown_ok'])}")
    print(f"Selection Mode: {best_search_row['selection_mode']}")
    print(f"Train Annual Return: {best_search_row['train_annual_return']:.4f}")
    print(f"Validation Annual Return: {best_search_row['validation_annual_return']:.4f}")
    print(f"Annual Return Gap: {annual_return_gap:.4f}")
    if str(best_search_row["selection_mode"]) == "fallback_validation_dd12":
        print("Constraint Status: 无满足验证回撤约束的候选")
    print(f"OOS Return Target: {TARGET_OOS_ANNUAL_RETURN:.4f}")
    print(f"OOS Drawdown Target: {TARGET_OOS_DRAWDOWN:.0%}")
    print(f"Return Target OK: {return_target_ok}")
    print(f"Drawdown Target OK: {drawdown_target_ok}")
    if not drawdown_target_ok:
        print(f"Drawdown Gap To Target: {drawdown_gap_to_target:.4f}")
    print(f"Full Training Annual Return: {train_metrics['annual_return']:.4f}")
    print(f"Train Sharpe: {train_metrics['sharpe_ratio']:.4f}")
    print(f"Test Annual Return: {test_metrics['annual_return']:.4f}")
    print(f"Test Sharpe: {test_metrics['sharpe_ratio']:.4f}")
    print(f"Test Max Drawdown: {test_metrics['max_drawdown']:.4f}")

    save_desktop_artifacts(
        best_params,
        local_grid_results,
        train_equity_df,
        train_trades_df,
        train_metrics,
        train_top_scored_targets,
        test_equity_df,
        test_trades_df,
        test_metrics,
        test_top_scored_targets,
    )
    print(f"Output Dir: {OUTPUT_DIR}")

    return best_params, local_grid_results, train_equity_df, test_equity_df


## 9. Load Data And Tune Parameters


In [ ]:
raw_data = {}
print("\nLoading ETF Data")
for symbol_name, symbol in TREND_ETF_POOL.items():
    if symbol not in THREE_CLASS_MAP:
        continue
    try:
        raw_data[symbol] = load_tushare_daily(symbol, START_DATE, END_DATE, source_type="fund")
        print(f"  Loaded {symbol_name} ({symbol})")
    except Exception as exc:
        print(f"  [WARN] {symbol_name} ({symbol}) load failed: {exc}")

benchmark_df = load_tushare_daily(BENCHMARK_CODE, START_DATE, END_DATE, source_type="fund")
print(f"\nSuccessfully loaded {len(raw_data)} ETFs")
print(f"Benchmark: {get_symbol_label(BENCHMARK_CODE)}")
print(f"Benchmark data: {len(benchmark_df)} days")

print("\nSTEP 1: Local Train/Validation Neighborhood Search")
best_params, local_grid_results = optimize_params_on_training_set(raw_data, benchmark_df)


## 10. Training Backtest


In [ ]:
print("\nSTEP 2: Training Set Backtest with Tuned Parameters")
train_equity_df, train_trades_df, train_metrics, train_top_scored_targets = run_backtest_on_period(
    best_params, raw_data, benchmark_df, "Training Set", START_DATE, TRAIN_END_DATE
)
train_metrics


## 11. Test Backtest


In [ ]:
print("\nSTEP 3: Test Set Backtest with Tuned Parameters (Out-of-Sample)")
test_equity_df, test_trades_df, test_metrics, test_top_scored_targets = run_backtest_on_period(
    best_params, raw_data, benchmark_df, "Test Set (OOS)", OOS_START_DATE, END_DATE
)
test_metrics


## 12. Save Artifacts


In [ ]:
RUN_TIMESTAMP = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "ETF_Result" / f"run_{RUN_TIMESTAMP}"
save_desktop_artifacts(
    best_params,
    local_grid_results,
    train_equity_df,
    train_trades_df,
    train_metrics,
    train_top_scored_targets,
    test_equity_df,
    test_trades_df,
    test_metrics,
    test_top_scored_targets,
)
print(f"Output Dir: {OUTPUT_DIR}")
